# 01 · Bangun Model Klasterisasi — FastDTW · 3 Representasi · 6 Algoritma

Notebook ini **membangun model** di Google Colab, lalu mengekspor **`hasil_model.zip`** untuk dashboard Streamlit (notebook `02_jalankan_streamlit`).

**Cara pakai**
1. Pastikan `Sales transaction (1).csv` ada di panel 📁 **Files** (pemisah `;` atau `,` dideteksi otomatis).
2. **Runtime → Run all**. Total ±5–10 menit (bagian 4 paling lama).
3. Di akhir, `hasil_model.zip` otomatis terunduh **dan** tersimpan di `/content` untuk notebook 02.

| Sesi | Isi |
|---|---|
| 1 | Membaca & memvalidasi CSV |
| 2 | Mesin analisis (`%%writefile`) + **konfigurasi** (bisa diubah) |
| 3 | EDA singkat + contoh tiga representasi |
| 4 | **Membangun model**: Data asli, Min-Max, Scale × FastDTW × K-Means DBA, K-Medoids PAM, Agglomerative, DBSCAN, Fuzzy C-Means DBA, Fuzzy C-Medoids |
| 5 | Evaluasi: model terbaik per representasi, stabilitas seed, ARI antar-representasi |
| 6 | Visual model terbaik (peta cluster, centroid, silhouette, profil) |
| 7 | Ekspor `hasil_model.zip` |

## Sesi 1 — Membaca dan memvalidasi data

In [ ]:
import os, sys, glob
import pandas as pd
from IPython.display import display

WORK = "/content/ruang_sales_model"
os.makedirs(f"{WORK}/scripts", exist_ok=True)
os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)

MINGGU = [f"W{i}" for i in range(52)]

def baca_csv(path):
    """Membaca CSV dengan pemisah titik koma atau koma (dideteksi otomatis)."""
    with open(path, encoding="utf-8-sig", errors="ignore") as f:
        baris = f.readline()
    sep = ";" if baris.count(";") > baris.count(",") else ","
    return pd.read_csv(path, sep=sep, encoding="utf-8-sig"), sep

CSV_PATH, raw, SEP = None, None, None
for path in sorted(glob.glob("/content/*.csv")):
    df, sep = baca_csv(path)
    df.columns = df.columns.astype(str).str.strip()
    if "Product_Code" in df.columns and set(MINGGU) <= set(df.columns):
        CSV_PATH, raw, SEP = path, df, sep
        break
assert CSV_PATH, "CSV dengan kolom Product_Code dan W0–W51 tidak ditemukan di /content. Seret file CSV ke panel 📁 Files."

sales = raw[["Product_Code", *MINGGU]].copy()
assert sales["Product_Code"].is_unique, "Product_Code ganda ditemukan."
assert not sales[MINGGU].isna().any().any(), "Ada nilai kosong pada W0–W51."
assert (sales[MINGGU] >= 0).all().all(), "Ada penjualan negatif."
print("✅ Dataset:", os.path.basename(CSV_PATH), "| pemisah:", "titik koma" if SEP == ";" else "koma")
print(f"   {len(sales)} produk × {len(MINGGU)} minggu · total penjualan {sales[MINGGU].to_numpy().sum():,.0f}")
print(f"   Kolom lain diabaikan: {raw.shape[1] - sales.shape[1]} (mis. MIN, MAX, Normalized 0–51)")
display(sales.head())


## Sesi 2 — Mesin analisis dan konfigurasi
`config.py` adalah satu-satunya tempat mengubah pengaturan (K, linkage, eps DBSCAN, guardrail). Sel lain tidak perlu diubah.

In [ ]:
%%writefile config.py
"""Konfigurasi terpusat: pembangunan model (Colab) dan dashboard (Streamlit)."""
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent
RESULTS_DIR = PROJECT_ROOT / "hasil_model"          # hasil dari notebook Colab (diekstrak dari hasil_model.zip)
RESULTS_FILE = RESULTS_DIR / "model_results.json"
CACHE_DIR = PROJECT_ROOT / "cache"                   # matriks jarak FastDTW (tidak perlu di-commit)

ID_COLUMN = "Product_Code"
WEEK_COLUMNS = [f"W{i}" for i in range(52)]

# ---------------- Pembangunan model (notebook Colab) ----------------
REPRESENTATIONS = ["original", "minmax", "zscore"]  # Data asli · Min-Max per produk · Scale (z-score per produk)
DISTANCE = "fastdtw"
FASTDTW_RADIUS = 2
K_GRID = [2, 3, 4, 5, 6]
LINKAGES = ["average", "complete"]
DBSCAN_MIN_SAMPLES = 7
DBSCAN_EPS_PERCENTILES = [30, 50, 70, 85, 95]        # persentil jarak tetangga ke-min_samples
FUZZINESS = 2.0
RANDOM_STATE = 42
KMEANS_N_INIT = 1

# Aturan memilih "model terbaik" per representasi (di atas peringkat Silhouette↑ · DBI↓ · CHI↑)
MIN_BALANCE = 0.50     # tolak partisi "satu cluster raksasa"
MAX_NOISE_SHARE = 0.30  # DBSCAN: maksimal 30% produk noise
MIN_CLUSTERS = 2

# ---------------- Dashboard ----------------
DEFAULT_REPRESENTATION = "minmax"

PLOT_FONT = "Plus Jakarta Sans, Arial"
PLOT_PRIMARY = "#00806A"
PLOT_ACCENT = "#E06C2B"
PLOT_PALETTE = ["#00806A", "#E06C2B", "#3A6FD9", "#C8457F", "#7B5BD6"]
PLOT_NOISE = "#9AA39E"
PLOT_INK = "#13201B"
PLOT_MUTED = "#6B7770"
PLOT_GRID = "#EEEBE2"
PLOT_SEQUENTIAL = ["#F5F3EC", "#BFE3D6", "#55B196", "#00806A", "#00483A"]
PLOT_SEQUENTIAL_WARM = ["#FBEFE7", "#F6CFB6", "#EDA47C", "#E06C2B", "#A9481A"]


In [ ]:
%%writefile scripts/__init__.py
"""Paket mesin analisis Ruang Sales."""


In [ ]:
%%writefile scripts/data.py
"""Fungsi pemuatan, validasi, dan ringkasan data penjualan mingguan."""
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd


def validate_sales_data(
    data: pd.DataFrame,
    id_column: str = "Product_Code",
    week_columns: list[str] | None = None,
    allow_missing: bool = False,
    allow_negative: bool = False,
) -> pd.DataFrame:
    """Validasi struktur, ID, dan nilai penjualan; mengembalikan salinan yang telah dirapikan."""
    if week_columns is None:
        week_columns = [f"W{i}" for i in range(52)]
    if not isinstance(data, pd.DataFrame) or data.empty:
        raise ValueError("Data penjualan harus berupa DataFrame yang tidak kosong.")

    frame = data.copy()
    frame.columns = frame.columns.astype(str).str.strip()
    required = [id_column, *week_columns]
    missing_columns = [column for column in required if column not in frame.columns]
    if missing_columns:
        raise ValueError(f"Kolom wajib tidak ditemukan: {missing_columns}")

    if frame[id_column].isna().any():
        raise ValueError(f"Kolom {id_column} mengandung ID kosong.")
    frame[id_column] = frame[id_column].astype(str).str.strip()
    if frame[id_column].eq("").any() or frame[id_column].duplicated().any():
        raise ValueError(f"{id_column} harus terisi dan unik untuk setiap produk.")

    for column in week_columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    values = frame[week_columns].to_numpy(dtype=float)
    if np.isinf(values).any():
        raise ValueError("Data penjualan mengandung nilai tak hingga.")
    if not allow_missing and np.isnan(values).any():
        locations = np.argwhere(np.isnan(values))[:10].tolist()
        raise ValueError(f"Data penjualan mengandung nilai kosong/teks. Contoh posisi: {locations}")
    if not allow_negative and np.nanmin(values) < 0:
        raise ValueError("Penjualan negatif ditemukan. Periksa apakah nilai tersebut retur/koreksi.")
    if allow_missing and np.isnan(values).any():
        medians = frame[week_columns].median(axis=0)
        frame[week_columns] = frame[week_columns].fillna(medians).fillna(0.0)
    return frame[[id_column, *week_columns]].reset_index(drop=True)


def load_sales_data(
    file_path: str | Path,
    id_column: str = "Product_Code",
    week_columns: list[str] | None = None,
    delimiter: str = ",",
    allow_missing: bool = False,
    allow_negative: bool = False,
) -> pd.DataFrame:
    """Membaca CSV penjualan dan menjalankan validasi skema serta nilai."""
    path = Path(file_path).expanduser()
    if not path.exists():
        raise FileNotFoundError(f"File data tidak ditemukan: {path.resolve()}")
    frame = pd.read_csv(path, sep=delimiter, dtype={id_column: "string"})
    return validate_sales_data(
        frame, id_column=id_column, week_columns=week_columns,
        allow_missing=allow_missing, allow_negative=allow_negative,
    )


def build_product_statistics(
    data: pd.DataFrame,
    id_column: str = "Product_Code",
    week_columns: list[str] | None = None,
) -> pd.DataFrame:
    """Membentuk statistik level produk dari penjualan asli selama seluruh minggu."""
    if week_columns is None:
        week_columns = [f"W{i}" for i in range(52)]
    values = data[week_columns].to_numpy(dtype=float)
    totals = values.sum(axis=1)
    active_count = (values > 0).sum(axis=1)
    peak_index = values.argmax(axis=1)
    return pd.DataFrame({
        "Produk": data[id_column].astype(str).to_numpy(),
        "Total volume": totals,
        "Rata-rata / minggu": values.mean(axis=1),
        "Minggu nol": (values == 0).sum(axis=1),
        "Minggu aktif (%)": 100.0 * active_count / values.shape[1],
        "Puncak mingguan": values.max(axis=1),
        "Minggu puncak": [f"W{i}" for i in peak_index],
    })


def build_weekly_statistics(
    data: pd.DataFrame,
    week_columns: list[str] | None = None,
) -> pd.DataFrame:
    """Meringkas volume total, median, dan persentase produk aktif pada setiap minggu."""
    if week_columns is None:
        week_columns = [f"W{i}" for i in range(52)]
    values = data[week_columns].to_numpy(dtype=float)
    return pd.DataFrame({
        "Minggu": np.arange(values.shape[1]),
        "Total_Penjualan": values.sum(axis=0),
        "Median_Penjualan_Produk": np.median(values, axis=0),
        "Persen_Produk_Aktif": 100.0 * (values > 0).mean(axis=0),
    })


In [ ]:
%%writefile scripts/distance.py
"""Implementasi FastDTW multiresolusi terakselerasi Numba dan utilitas matriks jarak."""
from __future__ import annotations

import hashlib
from pathlib import Path
import numpy as np
from numba import njit, prange, set_num_threads


@njit(cache=True)
def fastdtw_numba(a: np.ndarray, b: np.ndarray, radius: int = 2, return_path: bool = False):
    """Menghitung FastDTW univariat dengan biaya lokal kuadrat dan akar biaya akhir."""
    n0 = len(a)
    if n0 != len(b) or n0 == 0 or radius < 1:
        return np.inf, np.empty(0, dtype=np.int32), np.empty(0, dtype=np.int32)

    max_levels = 16
    aa = np.zeros((max_levels, n0), dtype=np.float64)
    bb = np.zeros((max_levels, n0), dtype=np.float64)
    lengths = np.zeros(max_levels, dtype=np.int32)
    aa[0, :] = a
    bb[0, :] = b
    lengths[0] = n0
    nlevels = 1

    while lengths[nlevels - 1] >= radius + 2 and nlevels < max_levels:
        m = lengths[nlevels - 1] // 2
        if m < 1:
            break
        for q in range(m):
            aa[nlevels, q] = (aa[nlevels - 1, 2 * q] + aa[nlevels - 1, 2 * q + 1]) / 2.0
            bb[nlevels, q] = (bb[nlevels - 1, 2 * q] + bb[nlevels - 1, 2 * q + 1]) / 2.0
        lengths[nlevels] = m
        nlevels += 1

    path_i = np.zeros(2 * n0 + 2, dtype=np.int32)
    path_j = np.zeros(2 * n0 + 2, dtype=np.int32)
    path_len = 0
    final_cost = np.inf

    for level in range(nlevels - 1, -1, -1):
        n = lengths[level]
        mask = np.zeros((n, n), dtype=np.bool_)
        if level == nlevels - 1:
            mask[:, :] = True
        else:
            for t in range(path_len):
                base_i = path_i[t]
                base_j = path_j[t]
                for di in range(-radius, radius + 1):
                    ii = base_i + di
                    if ii < 0 or ii >= lengths[level + 1]:
                        continue
                    for dj in range(-radius, radius + 1):
                        jj = base_j + dj
                        if jj < 0 or jj >= lengths[level + 1]:
                            continue
                        for ai in range(2):
                            x = ii * 2 + ai
                            if x >= n:
                                continue
                            for bj in range(2):
                                y = jj * 2 + bj
                                if y < n:
                                    mask[x, y] = True
            # Jaga agar titik awal dan akhir selalu berada dalam search window.
            mask[0, 0] = True
            mask[n - 1, n - 1] = True

        dp = np.full((n + 1, n + 1), np.inf, dtype=np.float64)
        move = np.zeros((n + 1, n + 1), dtype=np.int8)
        dp[0, 0] = 0.0
        for i in range(1, n + 1):
            for j in range(1, n + 1):
                if not mask[i - 1, j - 1]:
                    continue
                delta = aa[level, i - 1] - bb[level, j - 1]
                local_cost = delta * delta
                best = dp[i - 1, j]
                direction = 1
                if dp[i, j - 1] < best:
                    best = dp[i, j - 1]
                    direction = 2
                if dp[i - 1, j - 1] < best:
                    best = dp[i - 1, j - 1]
                    direction = 3
                dp[i, j] = best + local_cost
                move[i, j] = direction

        final_cost = dp[n, n]
        if not np.isfinite(final_cost):
            return np.inf, np.empty(0, dtype=np.int32), np.empty(0, dtype=np.int32)

        i, j = n, n
        p = 0
        while i > 0 and j > 0:
            path_i[p] = i - 1
            path_j[p] = j - 1
            p += 1
            direction = move[i, j]
            if direction == 1:
                i -= 1
            elif direction == 2:
                j -= 1
            elif direction == 3:
                i -= 1
                j -= 1
            else:
                return np.inf, np.empty(0, dtype=np.int32), np.empty(0, dtype=np.int32)

        # Jalur disimpan dari awal ke akhir.
        for t in range(p // 2):
            tmp_i = path_i[t]
            path_i[t] = path_i[p - 1 - t]
            path_i[p - 1 - t] = tmp_i
            tmp_j = path_j[t]
            path_j[t] = path_j[p - 1 - t]
            path_j[p - 1 - t] = tmp_j
        path_len = p

    distance = np.sqrt(final_cost)
    if return_path:
        return distance, path_i[:path_len].copy(), path_j[:path_len].copy()
    return distance, path_i[:0], path_j[:0]


@njit(parallel=True, cache=True)
def _all_pairs_fastdtw(X: np.ndarray, radius: int):
    """Kernel paralel untuk menghitung seluruh pasangan jarak antarpola."""
    n = X.shape[0]
    distances = np.zeros((n, n), dtype=np.float64)
    for i in prange(n):
        for j in range(i + 1, n):
            distance, _, _ = fastdtw_numba(X[i], X[j], radius, False)
            distances[i, j] = distance
            distances[j, i] = distance
    return distances


@njit(parallel=True, cache=True)
def _cross_fastdtw(X: np.ndarray, centers: np.ndarray, radius: int):
    """Kernel paralel untuk jarak setiap seri pada X ke setiap prototype."""
    output = np.zeros((len(X), len(centers)), dtype=np.float64)
    for i in prange(len(X)):
        for j in range(len(centers)):
            output[i, j] = fastdtw_numba(X[i], centers[j], radius, False)[0]
    return output


def exact_dtw_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Menghitung DTW eksak sebagai pemeriksaan pembanding pada seri pendek."""
    x = np.asarray(a, dtype=float).ravel()
    y = np.asarray(b, dtype=float).ravel()
    if x.size == 0 or y.size == 0:
        raise ValueError("Seri DTW tidak boleh kosong.")
    dp = np.full((len(x) + 1, len(y) + 1), np.inf, dtype=float)
    dp[0, 0] = 0.0
    for i in range(1, len(x) + 1):
        for j in range(1, len(y) + 1):
            local = (x[i - 1] - y[j - 1]) ** 2
            dp[i, j] = local + min(dp[i - 1, j], dp[i, j - 1], dp[i - 1, j - 1])
    return float(np.sqrt(dp[-1, -1]))


def compute_distance_matrix(X: np.ndarray, radius: int = 2, n_threads: int = 4) -> np.ndarray:
    """Menghitung matriks jarak FastDTW simetris untuk semua produk."""
    values = np.ascontiguousarray(X, dtype=np.float64)
    if values.ndim != 2 or values.shape[0] < 2 or values.shape[1] < 2:
        raise ValueError("X harus berupa matriks produk × minggu dengan minimal dua baris dan kolom.")
    if not np.isfinite(values).all():
        raise ValueError("X mengandung NaN atau inf.")
    if radius < 1:
        raise ValueError("radius FastDTW minimal 1.")
    set_num_threads(max(1, int(n_threads)))
    distances = _all_pairs_fastdtw(values, int(radius))
    np.fill_diagonal(distances, 0.0)
    if not np.isfinite(distances).all() or (distances < 0).any() or not np.allclose(distances, distances.T):
        raise RuntimeError("Matriks jarak FastDTW tidak valid.")
    return distances


def compute_cross_distance(X: np.ndarray, centers: np.ndarray, radius: int = 2) -> np.ndarray:
    """Menghitung jarak FastDTW antara kumpulan seri dan kumpulan prototype."""
    values = np.ascontiguousarray(X, dtype=np.float64)
    prototypes = np.ascontiguousarray(centers, dtype=np.float64)
    if values.ndim != 2 or prototypes.ndim != 2 or values.shape[1] != prototypes.shape[1]:
        raise ValueError("X dan centers harus memiliki jumlah minggu yang sama.")
    return _cross_fastdtw(values, prototypes, int(radius))


def data_digest(X: np.ndarray) -> str:
    """Menghasilkan sidik data singkat untuk validasi cache matriks jarak."""
    values = np.ascontiguousarray(X, dtype="<f8")
    return hashlib.sha256(values.tobytes()).hexdigest()[:16]


def load_or_compute_distance_matrix(
    X: np.ndarray,
    radius: int,
    cache_dir: str | Path,
    reuse_cache: bool = True,
    n_threads: int = 4,
) -> tuple[np.ndarray, Path, bool]:
    """Memuat cache matriks FastDTW yang cocok atau menghitung dan menyimpannya."""
    folder = Path(cache_dir)
    folder.mkdir(parents=True, exist_ok=True)
    digest = data_digest(X)
    cache_path = folder / f"fastdtw_{digest}_r{int(radius)}.npy"
    if reuse_cache and cache_path.exists():
        cached = np.load(cache_path, mmap_mode=None)
        expected = (len(X), len(X))
        valid = cached.shape == expected and np.isfinite(cached).all()
        valid = valid and np.allclose(cached, cached.T) and np.allclose(np.diag(cached), 0.0)
        if valid:
            return cached, cache_path, True
        cache_path.unlink()
    distances = compute_distance_matrix(X, radius=radius, n_threads=n_threads)
    np.save(cache_path, distances)
    return distances, cache_path, False


In [ ]:
%%writefile scripts/interactive.py
"""Mesin klasterisasi on-demand untuk dashboard (gaya R Shiny: pilih → jalankan → tampil).

Modul ini sengaja *tidak* bergantung pada Streamlit maupun numba sehingga bisa dipakai di
notebook, di dashboard, dan diuji terpisah. Semua perhitungan DTW memakai NumPy yang
divektorisasi di sepanjang pasangan deret (cepat untuk 811 × 52 tanpa kompilasi JIT).

Kombinasi yang didukung
-----------------------
Representasi : ``original`` (data asli), ``minmax`` (Min-Max per produk), ``zscore`` (scale/z-score per produk)
Jarak        : ``euclidean``, ``dtw`` (penuh), ``cdtw`` (pita Sakoe-Chiba), ``fastdtw`` (pakai numba bila ada)
Algoritma    : ``kmeans`` (rata-rata / DBA), ``kmedoids`` (PAM BUILD+SWAP), ``agglomerative``,
               ``dbscan`` (eps otomatis dari k-distance), ``fcm`` (Fuzzy C-Means / DBA), ``fcmdd`` (Fuzzy C-Medoids)

Definisi "centroid" yang ditampilkan
------------------------------------
* K-Means / FCM + Euclidean → rata-rata aritmetika anggota (pusat Lloyd).
* K-Means / FCM + DTW       → barycenter DBA (DTW Barycenter Averaging).
* K-Medoids / FCMdd         → medoid (produk nyata).
* Agglomerative / DBSCAN    → tidak punya pusat bawaan; ditampilkan medoid + rata-rata anggota.
"""
from __future__ import annotations

import hashlib
import time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_samples

REPRESENTATIONS = {
    "original": "Data asli",
    "minmax": "Min-Max per produk",
    "zscore": "Scale (z-score per produk)",
}
DISTANCES = {
    "euclidean": "Euclidean",
    "dtw": "DTW",
    "cdtw": "CDTW (Sakoe-Chiba)",
    "fastdtw": "FastDTW",
}
ALGORITHMS = {
    "kmeans": "K-Means (DBA untuk DTW)",
    "kmedoids": "K-Medoids PAM",
    "agglomerative": "Hierarchical (Agglomerative)",
    "dbscan": "DBSCAN",
    "fcm": "Fuzzy C-Means",
    "fcmdd": "Fuzzy C-Medoids",
}
CENTER_KIND = {"mean": "Rata-rata anggota", "dba": "Barycenter DBA", "medoid": "Medoid (produk nyata)"}


# =======================================================================================
# Representasi data
# =======================================================================================
def represent(W: np.ndarray, kind: str) -> np.ndarray:
    """Mengubah matriks penjualan (produk × minggu) ke representasi terpilih (row-wise)."""
    values = np.asarray(W, dtype=np.float64)
    if kind == "original":
        return values.copy()
    if kind == "minmax":
        low, high = values.min(axis=1, keepdims=True), values.max(axis=1, keepdims=True)
        span = high - low
        return np.where(span > 0, (values - low) / np.where(span > 0, span, 1.0), 0.0)
    if kind == "zscore":
        mean, std = values.mean(axis=1, keepdims=True), values.std(axis=1, keepdims=True)
        return np.where(std > 0, (values - mean) / np.where(std > 0, std, 1.0), 0.0)
    raise ValueError(f"Representasi tidak dikenal: {kind}")


# =======================================================================================
# DTW tervektorisasi
# =======================================================================================
def _band(i: int, m: int, window: int | None) -> tuple[int, int]:
    """Rentang kolom j (1-based, inklusif) yang boleh dikunjungi pada baris i."""
    if window is None:
        return 1, m
    return max(1, i - window), min(m, i + window)


def dtw_batch(A: np.ndarray, B: np.ndarray, window: int | None = None, return_cost: bool = False):
    """DTW untuk P pasangan sekaligus. A: (P, n), B: (P, m).

    Biaya lokal = selisih kuadrat; jarak akhir = sqrt(jumlah biaya pada jalur optimal),
    konsisten dengan definisi FastDTW di ``scripts/distance.py``.
    Jika ``return_cost`` True, juga mengembalikan matriks biaya kumulatif (P, n+1, m+1).
    """
    A = np.asarray(A, dtype=np.float64)
    B = np.asarray(B, dtype=np.float64)
    P, n = A.shape
    m = B.shape[1]
    if window is not None:
        window = max(int(window), abs(n - m))
    if return_cost:
        D = np.full((P, n + 1, m + 1), np.inf)
        D[:, 0, 0] = 0.0
        for i in range(1, n + 1):
            lo, hi = _band(i, m, window)
            cost = (A[:, i - 1, None] - B[:, lo - 1:hi]) ** 2
            prev, cur = D[:, i - 1], D[:, i]
            for j in range(lo, hi + 1):
                cur[:, j] = cost[:, j - lo] + np.minimum(np.minimum(prev[:, j], prev[:, j - 1]), cur[:, j - 1])
        return np.sqrt(D[:, n, m]), D
    # Tata letak (kolom, pasangan) agar setiap langkah j membaca memori bersebelahan.
    At, Bt = np.ascontiguousarray(A.T), np.ascontiguousarray(B.T)
    prev = np.full((m + 1, P), np.inf)
    prev[0] = 0.0
    tmp = np.empty(P)
    for i in range(1, n + 1):
        cur = np.full((m + 1, P), np.inf)
        lo, hi = _band(i, m, window)
        cost = (Bt[lo - 1:hi] - At[i - 1]) ** 2
        for j in range(lo, hi + 1):
            np.minimum(prev[j], prev[j - 1], out=tmp)
            np.minimum(tmp, cur[j - 1], out=tmp)
            np.add(tmp, cost[j - lo], out=cur[j])
        prev = cur
    return np.sqrt(prev[m])


def dtw_paths(D: np.ndarray) -> list[np.ndarray]:
    """Backtracking jalur optimal dari matriks biaya kumulatif (P, n+1, m+1).

    Mengembalikan list berisi array (L, 2) indeks (i, j) 0-based untuk setiap pasangan.
    """
    P, n1, m1 = D.shape
    i = np.full(P, n1 - 1)
    j = np.full(P, m1 - 1)
    steps_i, steps_j, active_steps = [], [], []
    rows = np.arange(P)
    active = np.ones(P, dtype=bool)
    while active.any():
        steps_i.append(i - 1)
        steps_j.append(j - 1)
        active_steps.append(active.copy())
        diag = D[rows, np.maximum(i - 1, 0), np.maximum(j - 1, 0)]
        up = D[rows, np.maximum(i - 1, 0), j]
        left = D[rows, i, np.maximum(j - 1, 0)]
        choice = np.argmin(np.stack([diag, up, left]), axis=0)
        new_i = np.where(choice == 2, i, i - 1)
        new_j = np.where(choice == 1, j, j - 1)
        i = np.where(active, new_i, i)
        j = np.where(active, new_j, j)
        active = active & ~((i == 0) & (j == 0)) & (i > 0) & (j > 0)
    si, sj, sa = np.array(steps_i), np.array(steps_j), np.array(active_steps)
    return [np.c_[si[sa[:, p], p], sj[sa[:, p], p]][::-1] for p in range(P)]


def dtw_pair(a: np.ndarray, b: np.ndarray, window: int | None = None) -> tuple[float, np.ndarray, np.ndarray]:
    """DTW satu pasangan: (jarak, jalur (L,2), matriks biaya kumulatif tanpa baris/kolom 0)."""
    distance, D = dtw_batch(np.asarray(a)[None], np.asarray(b)[None], window, return_cost=True)
    return float(distance[0]), dtw_paths(D)[0], D[0, 1:, 1:]


def pairwise_dtw(X: np.ndarray, window: int | None = None, chunk: int = 40000, progress=None) -> np.ndarray:
    """Matriks jarak DTW/CDTW simetris untuk seluruh produk (diproses per blok pasangan)."""
    X = np.asarray(X, dtype=np.float64)
    n = len(X)
    rows, cols = np.triu_indices(n, k=1)
    values = np.empty(len(rows))
    for start in range(0, len(rows), chunk):
        stop = min(start + chunk, len(rows))
        values[start:stop] = dtw_batch(X[rows[start:stop]], X[cols[start:stop]], window)
        if progress:
            progress(stop / len(rows))
    D = np.zeros((n, n))
    D[rows, cols] = values
    D[cols, rows] = values
    return D


def cross_dtw(X: np.ndarray, centers: np.ndarray, window: int | None = None) -> np.ndarray:
    """Jarak DTW setiap deret ke setiap pusat: (n, k)."""
    n, k = len(X), len(centers)
    A = np.repeat(np.asarray(X, dtype=np.float64), k, axis=0)
    B = np.tile(np.asarray(centers, dtype=np.float64), (n, 1))
    return dtw_batch(A, B, window).reshape(n, k)


def dba_update(X: np.ndarray, center: np.ndarray, weights: np.ndarray | None = None,
               window: int | None = None, iterations: int = 3) -> np.ndarray:
    """DTW Barycenter Averaging berbobot (Petitjean dkk., 2011), divektorisasi per iterasi."""
    X = np.asarray(X, dtype=np.float64)
    center = np.asarray(center, dtype=np.float64).copy()
    w = np.ones(len(X)) if weights is None else np.asarray(weights, dtype=np.float64)
    keep = w > 1e-10
    X, w = X[keep], w[keep]
    if len(X) == 0:
        return center
    for _ in range(int(iterations)):
        _, D = dtw_batch(np.repeat(center[None], len(X), axis=0), X, window, return_cost=True)
        sums = np.zeros_like(center)
        counts = np.zeros_like(center)
        for p, path in enumerate(dtw_paths(D)):
            np.add.at(sums, path[:, 0], w[p] * X[p, path[:, 1]])
            np.add.at(counts, path[:, 0], w[p])
        valid = counts > 1e-12
        center[valid] = sums[valid] / counts[valid]
    return center


def distance_matrix(X: np.ndarray, metric: str, window: int = 5, radius: int = 2,
                    cache_dir: Path | None = None, progress=None) -> tuple[np.ndarray, str]:
    """Matriks jarak penuh + catatan. Disimpan ke cache disk bila ``cache_dir`` diberikan."""
    X = np.asarray(X, dtype=np.float64)
    tag = {"euclidean": "", "dtw": "", "cdtw": f"_w{window}", "fastdtw": f"_r{radius}"}[metric]
    digest = hashlib.sha1(np.ascontiguousarray(X).tobytes()).hexdigest()[:16]
    path = Path(cache_dir) / f"interactive_{metric}{tag}_{digest}.npy" if cache_dir else None
    if path is not None and path.is_file():
        return np.load(path), "dimuat dari cache"
    note = ""
    if metric == "euclidean":
        D = squareform(pdist(X, metric="euclidean"))
    elif metric == "dtw":
        D = pairwise_dtw(X, None, progress=progress)
    elif metric == "cdtw":
        D = pairwise_dtw(X, int(window), progress=progress)
    elif metric == "fastdtw":
        legacy = Path(cache_dir) / f"fastdtw_{hashlib.sha256(np.ascontiguousarray(X, dtype=np.float64).tobytes()).hexdigest()[:16]}_r{radius}.npy" if cache_dir else None
        if legacy is not None and legacy.is_file():
            D, note = np.load(legacy), "memakai matriks FastDTW notebook"
        else:
            try:
                import numba
                from scripts.distance import compute_distance_matrix  # numba
                threads = int(getattr(numba.config, "NUMBA_NUM_THREADS", 1))  # Colab/Cloud hanya 2 core
                D, note = compute_distance_matrix(X, radius=radius, n_threads=threads), "FastDTW numba"
            except Exception:  # numba tidak tersedia → aproksimasi CDTW dengan pita sebesar radius
                D = pairwise_dtw(X, max(1, int(radius)), progress=progress)
                note = f"numba tidak tersedia; FastDTW diaproksimasi CDTW pita {radius}"
    else:
        raise ValueError(f"Jarak tidak dikenal: {metric}")
    if path is not None:
        try:
            path.parent.mkdir(parents=True, exist_ok=True)
            np.save(path, D)
        except OSError:
            pass
    return D, note or "dihitung"


# =======================================================================================
# Proyeksi 2D (MDS klasik) untuk visual fviz_cluster
# =======================================================================================
@dataclass
class Embedding:
    """Hasil MDS klasik: koordinat, nilai eigen, dan bahan proyeksi titik baru."""
    coords: np.ndarray            # (n, d)
    explained: np.ndarray         # (d,) rasio varians
    eigvals: np.ndarray           # (d,)
    eigvecs: np.ndarray           # (n, d)
    col_mean_sq: np.ndarray       # (n,) rerata kolom D²

    def project(self, distances_to_points: np.ndarray) -> np.ndarray:
        """Proyeksi titik baru (mis. barycenter DBA) dengan formula Gower dari jaraknya ke semua titik."""
        d2 = np.asarray(distances_to_points, dtype=float) ** 2
        b = -0.5 * (d2 - self.col_mean_sq[None, :] if d2.ndim == 2 else d2 - self.col_mean_sq)
        return (b @ self.eigvecs) / np.sqrt(self.eigvals)


def classical_mds(D: np.ndarray, dims: int = 5) -> Embedding:
    """MDS klasik (Torgerson). Untuk jarak Euclidean hasilnya identik dengan PCA."""
    D2 = np.asarray(D, dtype=float) ** 2
    n = len(D2)
    J = np.eye(n) - 1.0 / n
    Bm = -0.5 * J @ D2 @ J
    values, vectors = np.linalg.eigh(Bm)
    order = np.argsort(values)[::-1]
    values, vectors = values[order], vectors[:, order]
    positive = values[values > 1e-9]
    keep = min(dims, len(positive))
    lam, vec = values[:keep], vectors[:, :keep]
    return Embedding(
        coords=vec * np.sqrt(lam), explained=lam / positive.sum(), eigvals=lam, eigvecs=vec,
        col_mean_sq=D2.mean(axis=0) - D2.mean(),
    )


# =======================================================================================
# Algoritma
# =======================================================================================
def _init_pp(D: np.ndarray, k: int, seed: int) -> np.ndarray:
    """Inisialisasi k-means++ berbasis jarak (sama dengan ``scripts/clustering.py``)."""
    rng = np.random.default_rng(seed)
    n = len(D)
    chosen = [int(rng.integers(n))]
    while len(chosen) < k:
        weights = np.min(D[:, chosen], axis=1) ** 2
        weights[chosen] = 0.0
        chosen.append(int(rng.choice(n, p=weights / weights.sum())) if weights.sum() > 0
                      else int(rng.choice(np.setdiff1d(np.arange(n), chosen))))
    return np.asarray(chosen, dtype=int)


def _memberships(dist: np.ndarray, m: float) -> np.ndarray:
    """Keanggotaan fuzzy dari jarak ke pusat (u_ij ∝ d_ij^(-2/(m-1)))."""
    d = np.maximum(np.asarray(dist, dtype=float), 1e-12)
    inv = d ** (-2.0 / (m - 1.0))
    return inv / inv.sum(axis=1, keepdims=True)


def pam(D: np.ndarray, k: int, max_iter: int = 100) -> tuple[np.ndarray, np.ndarray, int]:
    """K-Medoids PAM klasik: BUILD rakus + SWAP terbaik per iterasi (tervektorisasi)."""
    n = len(D)
    medoids = [int(np.argmin(D.sum(axis=1)))]
    for _ in range(1, k):
        nearest = D[:, medoids].min(axis=1)
        gain = np.maximum(nearest[:, None] - D, 0).sum(axis=0)
        gain[medoids] = -np.inf
        medoids.append(int(np.argmax(gain)))
    medoids = np.asarray(medoids)
    iteration = 0
    for iteration in range(1, max_iter + 1):
        dm = D[:, medoids]
        order = np.argsort(dm, axis=1)
        assign = order[:, 0]
        d1 = dm[np.arange(n), assign]
        d2 = dm[np.arange(n), order[:, 1]] if k > 1 else np.full(n, np.inf)
        base = np.minimum(D - d1[:, None], 0).sum(axis=0)        # (h,) bagian dari titik di luar cluster i
        best_delta, best = 0.0, None
        for i in range(k):
            mask = assign == i
            Di = D[mask]                                          # (|Ci|, n)
            own = (np.minimum(Di, d2[mask, None]) - d1[mask, None]).sum(axis=0)
            correction = np.minimum(Di - d1[mask, None], 0).sum(axis=0)
            delta = base - correction + own
            delta[medoids] = np.inf
            h = int(np.argmin(delta))
            if delta[h] < best_delta - 1e-10:
                best_delta, best = float(delta[h]), (i, h)
        if best is None:
            break
        medoids[best[0]] = best[1]
    labels = np.argmin(D[:, medoids], axis=1)
    return labels, medoids, iteration


def _kmeans_dtw(X, D, k, window, seed, n_init, max_iter=15, dba_iter=3):
    """K-Means dengan barycenter DBA (jarak DTW/CDTW)."""
    best = None
    for run in range(max(1, n_init)):
        centers = X[_init_pp(D, k, seed + run)].copy()
        previous = None
        for it in range(max_iter):
            dist = cross_dtw(X, centers, window)
            labels = dist.argmin(axis=1)
            if previous is not None and np.array_equal(labels, previous):
                break
            previous = labels
            for c in range(k):
                members = labels == c
                if members.any():
                    centers[c] = dba_update(X[members], centers[c], None, window, dba_iter)
                else:
                    centers[c] = X[dist.min(axis=1).argmax()]
        dist = cross_dtw(X, centers, window)
        labels = dist.argmin(axis=1)
        objective = float((dist.min(axis=1) ** 2).sum())
        if best is None or objective < best[0]:
            best = (objective, labels, centers.copy(), dist, it + 1)
    return best


def _fcm_dtw(X, D, k, m, window, seed, max_iter=8, dba_iter=2, tol=1e-4):
    """Fuzzy C-Means dengan pusat DBA berbobot u^m."""
    centers = X[_init_pp(D, k, seed)].copy()
    it = 0
    for it in range(1, max_iter + 1):
        U = _memberships(cross_dtw(X, centers, window), m)
        updated = np.stack([dba_update(X, centers[c], U[:, c] ** m, window, dba_iter) for c in range(k)])
        shift = float(np.abs(updated - centers).max())
        centers = updated
        if shift < tol:
            break
    dist = cross_dtw(X, centers, window)
    return _memberships(dist, m), centers, dist, it


def _fcm_euclid(X, k, m, seed, max_iter=150, tol=1e-6):
    """Fuzzy C-Means klasik (Bezdek) pada ruang Euclidean."""
    rng = np.random.default_rng(seed)
    U = rng.dirichlet(np.ones(k), size=len(X))
    it = 0
    for it in range(1, max_iter + 1):
        Um = U ** m
        centers = (Um.T @ X) / Um.sum(axis=0)[:, None]
        dist = np.sqrt(((X[:, None, :] - centers[None]) ** 2).sum(axis=2))
        U_new = _memberships(dist, m)
        if np.abs(U_new - U).max() < tol:
            U = U_new
            break
        U = U_new
    return U, centers, dist, it


def _fcmdd(D, k, m, seed, max_iter=30):
    """Fuzzy C-Medoids (Krishnapuram dkk.) dengan medoid berbobot keanggotaan."""
    medoids = _init_pp(D, k, seed)
    D2 = D ** 2
    it = 0
    for it in range(1, max_iter + 1):
        U = _memberships(D[:, medoids], m)
        scores = D2.T @ (U ** m)
        proposed = []
        for c in range(k):
            for cand in np.argsort(scores[:, c]):
                if int(cand) not in proposed:
                    proposed.append(int(cand))
                    break
        proposed = np.asarray(proposed)
        if np.array_equal(proposed, medoids):
            break
        medoids = proposed
    return _memberships(D[:, medoids], m), medoids, it


def kdistance(D: np.ndarray, k: int) -> np.ndarray:
    """Jarak ke tetangga ke-k setiap titik, terurut naik (untuk plot kNNdist gaya R dbscan)."""
    return np.sort(np.sort(D, axis=1)[:, int(k)])


@dataclass
class ClusterResult:
    """Semua keluaran satu kali jalankan, siap untuk visual dan tabel."""
    params: dict
    labels: np.ndarray
    centers: np.ndarray                 # (k, T) pusat pada ruang representasi
    center_kind: str                    # mean | dba | medoid
    medoids: np.ndarray                 # indeks medoid per cluster (urut cluster)
    cluster_ids: np.ndarray             # ID cluster non-noise terurut
    dist_to_center: np.ndarray          # (n,) jarak produk ke pusat cluster-nya (NaN untuk noise)
    center_dist: np.ndarray             # (n, k) jarak tiap produk ke tiap pusat
    membership: np.ndarray | None = None
    linkage: np.ndarray | None = None
    eps: float | None = None
    n_iter: int = 0
    runtime: float = 0.0
    notes: list[str] = field(default_factory=list)
    metrics: dict = field(default_factory=dict)
    silhouette: np.ndarray | None = None
    center_xy: np.ndarray | None = None  # (k, 2) posisi pusat di peta MDS


def _medoids_of(D: np.ndarray, labels: np.ndarray, ids: np.ndarray) -> np.ndarray:
    out = []
    for c in ids:
        members = np.flatnonzero(labels == c)
        out.append(int(members[np.argmin(D[np.ix_(members, members)].sum(axis=1))]))
    return np.asarray(out, dtype=int)


def run_clustering(
    X: np.ndarray,
    D: np.ndarray,
    algorithm: str,
    metric: str,
    k: int = 2,
    linkage_method: str = "average",
    min_samples: int = 7,
    eps_percentile: float = 85.0,
    eps: float | None = None,
    fuzziness: float = 2.0,
    window: int = 5,
    seed: int = 42,
    n_init: int = 3,
) -> ClusterResult:
    """Menjalankan satu algoritma pada representasi X dan matriks jarak D."""
    start = time.perf_counter()
    X = np.asarray(X, dtype=np.float64)
    n = len(X)
    euclid = metric == "euclidean"
    win = None if metric in ("dtw", "fastdtw") else (int(window) if metric == "cdtw" else None)
    notes: list[str] = []
    membership = linkage_matrix = used_eps = None
    n_iter = 0
    if algorithm == "kmeans":
        if euclid:
            model = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(X)
            labels, centers, n_iter = model.labels_, model.cluster_centers_, int(model.n_iter_)
            center_dist = np.sqrt(((X[:, None, :] - centers[None]) ** 2).sum(axis=2))
            kind = "mean"
        else:
            if metric == "fastdtw":
                notes.append("Pembaruan pusat DBA memakai DTW penuh (FastDTW hanya untuk matriks jarak).")
            _, labels, centers, center_dist, n_iter = _kmeans_dtw(X, D, k, win, seed, n_init)
            kind = "dba"
    elif algorithm == "kmedoids":
        labels, medoids, n_iter = pam(D, k)
        centers, kind, center_dist = X[medoids], "medoid", D[:, medoids]
    elif algorithm == "agglomerative":
        if linkage_method == "ward" and not euclid:
            notes.append("Ward hanya sah untuk jarak Euclidean; dipakai 'average'.")
            linkage_method = "average"
        linkage_matrix = linkage(squareform(D, checks=False), method=linkage_method)
        raw = fcluster(linkage_matrix, t=k, criterion="maxclust")
        _, labels = np.unique(raw, return_inverse=True)
        kind = "medoid"
    elif algorithm == "dbscan":
        used_eps = float(eps) if eps else float(np.percentile(np.sort(D, axis=1)[:, int(min_samples)], eps_percentile))
        labels = DBSCAN(eps=used_eps, min_samples=int(min_samples), metric="precomputed").fit_predict(D)
        kind = "medoid"
        notes.append(f"eps = {used_eps:.4g} (persentil ke-{eps_percentile:g} jarak tetangga ke-{min_samples})" if not eps else f"eps manual = {used_eps:.4g}")
    elif algorithm == "fcm":
        if euclid:
            membership, centers, center_dist, n_iter = _fcm_euclid(X, k, fuzziness, seed)
            kind = "mean"
        else:
            membership, centers, center_dist, n_iter = _fcm_dtw(X, D, k, fuzziness, win, seed)
            kind = "dba"
        labels = membership.argmax(axis=1)
    elif algorithm == "fcmdd":
        membership, medoids, n_iter = _fcmdd(D, k, fuzziness, seed)
        labels = membership.argmax(axis=1)
        centers, kind, center_dist = X[medoids], "medoid", D[:, medoids]
    else:
        raise ValueError(f"Algoritma tidak dikenal: {algorithm}")

    labels = np.asarray(labels, dtype=int)
    ids = np.array(sorted(set(labels.tolist()) - {-1}), dtype=int)
    if algorithm in ("agglomerative", "dbscan"):
        medoids = _medoids_of(D, labels, ids) if len(ids) else np.array([], dtype=int)
        centers = X[medoids] if len(ids) else np.empty((0, X.shape[1]))
        center_dist = D[:, medoids] if len(ids) else np.empty((n, 0))
    else:
        if algorithm in ("kmeans", "fcm"):
            # cluster bisa kosong → buang pusat yatim agar indeks konsisten
            used = np.isin(np.arange(len(centers)), ids)
            remap = -np.ones(len(centers), dtype=int)
            remap[used] = np.arange(used.sum())
            centers, center_dist = centers[used], center_dist[:, used]
            if membership is not None:
                membership = membership[:, used] / membership[:, used].sum(axis=1, keepdims=True)
            labels = np.where(labels >= 0, remap[np.maximum(labels, 0)], -1)
            ids = np.arange(used.sum())
            medoids = _medoids_of(D, labels, ids)  # produk representatif (terdekat ke semua anggota)
    own = np.full(n, np.nan)
    for pos, c in enumerate(ids):
        mask = labels == c
        own[mask] = center_dist[mask, pos]
    return ClusterResult(
        params=dict(algorithm=algorithm, metric=metric, k=k, linkage=linkage_method, min_samples=min_samples,
                    eps_percentile=eps_percentile, fuzziness=fuzziness, window=window, seed=seed),
        labels=labels, centers=np.asarray(centers), center_kind=kind, medoids=medoids, cluster_ids=ids,
        dist_to_center=own, center_dist=np.asarray(center_dist), membership=membership, linkage=linkage_matrix,
        eps=used_eps, n_iter=int(n_iter), runtime=time.perf_counter() - start, notes=notes,
    )


# =======================================================================================
# Evaluasi
# =======================================================================================
def medoid_dbi(D: np.ndarray, labels: np.ndarray) -> float:
    """Davies–Bouldin adaptasi berbasis medoid pada matriks jarak apa pun."""
    ids = sorted(set(labels.tolist()) - {-1})
    if len(ids) < 2:
        return float("nan")
    med = _medoids_of(D, labels, np.asarray(ids))
    scatter = np.array([D[labels == c, m].mean() for c, m in zip(ids, med)])
    between = D[np.ix_(med, med)].copy()
    np.fill_diagonal(between, np.inf)
    ratio = (scatter[:, None] + scatter[None, :]) / np.maximum(between, 1e-12)
    np.fill_diagonal(ratio, -np.inf)
    return float(ratio.max(axis=1).mean())


def balance_score(labels: np.ndarray) -> float:
    """Entropi ukuran cluster ternormalisasi: 1 = ukuran seimbang, 0 = satu cluster dominan."""
    ids, counts = np.unique(labels[labels != -1], return_counts=True)
    if len(ids) < 2:
        return 0.0
    p = counts / counts.sum()
    return float(-(p * np.log(p)).sum() / np.log(len(ids)))


def evaluate(result: ClusterResult, D: np.ndarray, embedding: Embedding, W: np.ndarray) -> ClusterResult:
    """Mengisi metrik kualitas, Silhouette per produk, dan posisi pusat di peta MDS."""
    y = result.labels
    covered = y != -1
    ids = result.cluster_ids
    metrics = {
        "Silhouette": np.nan, "DBI_medoid": np.nan, "DBI_MDS": np.nan, "CHI_MDS": np.nan,
        "Balance": balance_score(y), "Coverage": float(covered.mean()), "Noise": int((~covered).sum()),
        "Cluster": int(len(ids)), "Eta2_nol": np.nan, "Waktu_s": result.runtime,
        "Within": float(np.nansum(result.dist_to_center)),
    }
    zeros = (np.asarray(W)[covered] == 0).sum(axis=1)
    if covered.any() and zeros.var() > 0:
        groups = y[covered]
        between = sum((groups == c).sum() * (zeros[groups == c].mean() - zeros.mean()) ** 2 for c in ids)
        metrics["Eta2_nol"] = float(between / ((zeros - zeros.mean()) ** 2).sum())
    silhouette = np.full(len(y), np.nan)
    if 2 <= len(ids) < covered.sum():
        keep = np.flatnonzero(covered)
        silhouette[keep] = silhouette_samples(D[np.ix_(keep, keep)], y[keep], metric="precomputed")
        metrics["Silhouette"] = float(np.nanmean(silhouette))
        metrics["DBI_medoid"] = medoid_dbi(D, y)
        emb = embedding.coords[keep]
        metrics["DBI_MDS"] = float(davies_bouldin_score(emb, y[keep]))
        metrics["CHI_MDS"] = float(calinski_harabasz_score(emb, y[keep]))
    if result.membership is not None:
        U = result.membership
        metrics["FPC"] = float((U ** 2).sum() / len(U))
        metrics["Entropi_partisi"] = float(-(U * np.log(np.maximum(U, 1e-15))).sum() / len(U))
    if result.center_kind == "medoid" or len(ids) == 0:
        xy = embedding.coords[result.medoids, :2] if len(ids) else np.empty((0, 2))
    else:
        xy = embedding.project(result.center_dist.T)[:, :2]
    result.metrics, result.silhouette, result.center_xy = metrics, silhouette, xy
    return result


def nbclust(X, D, embedding, W, algorithm, metric, k_values=range(2, 9), **kwargs) -> list[dict]:
    """Kurva siku (within) dan rata-rata Silhouette untuk beberapa k (gaya fviz_nbclust)."""
    rows = []
    for k in k_values:
        res = evaluate(run_clustering(X, D, algorithm, metric, k=int(k), **kwargs), D, embedding, W)
        rows.append({"k": int(k), **res.metrics})
    return rows


def rank_methods(table: list[dict]) -> list[dict]:
    """Peringkat rata-rata Silhouette↑, DBI(MDS)↓, CHI(MDS)↑ dengan bobot sama (seperti app referensi)."""
    import pandas as pd
    frame = pd.DataFrame(table)
    frame["Rank_Silhouette"] = frame["Silhouette"].rank(ascending=False, method="min")
    frame["Rank_DBI"] = frame["DBI_MDS"].rank(ascending=True, method="min")
    frame["Rank_CHI"] = frame["CHI_MDS"].rank(ascending=False, method="min")
    frame["Skor_Ranking"] = frame[["Rank_Silhouette", "Rank_DBI", "Rank_CHI"]].mean(axis=1)
    return frame.sort_values(["Skor_Ranking", "Silhouette"], ascending=[True, False]).to_dict("records")


def cluster_profile_rows(result: ClusterResult, W: np.ndarray, codes: np.ndarray) -> list[dict]:
    """Profil bisnis per cluster (data asli) + interpretasi tren 13 minggu awal vs akhir."""
    W = np.asarray(W, dtype=float)
    total = W.sum()
    rows = []
    groups = list(result.cluster_ids) + ([-1] if (result.labels == -1).any() else [])
    for c in groups:
        mask = result.labels == c
        G = W[mask]
        profile = G.mean(axis=0)
        first, last = profile[:13].mean(), profile[-13:].mean()
        change = 100 * (last - first) / first if first > 0 else np.nan
        cv = float(profile.std() / profile.mean()) if profile.mean() > 0 else np.nan
        zero_median = float(np.median((G == 0).sum(axis=1)))
        if c == -1:
            trend, action = "Noise", "Tinjau manual; pola tidak cukup dekat dengan cluster mana pun."
        elif np.isnan(change) or abs(change) < 5:
            trend, action = "Stabil", "Pertahankan kebijakan stok; pantau deviasi mingguan."
        elif change < 0:
            trend, action = "Tren melemah", "Evaluasi reorder point dan hindari penambahan stok sebelum penurunan terkonfirmasi."
        else:
            trend, action = "Tren menguat", "Tinjau kenaikan kebutuhan dan pertimbangkan penyesuaian safety stock secara bertahap."
        if zero_median >= 30 and c != -1:
            persona = "Sporadis"
        elif zero_median <= 4 and c != -1:
            persona = "Aktif & konsisten"
        elif c != -1:
            persona = "Fluktuatif"
        else:
            persona = "Tak terkelompok"
        pos = int(np.flatnonzero(result.cluster_ids == c)[0]) if c != -1 else None
        rows.append({
            "Cluster": int(c), "Persona": persona, "Tren": trend, "Rekomendasi": action,
            "Jumlah_Produk": int(mask.sum()), "Proporsi_%": 100 * float(mask.mean()),
            "Total_Penjualan": float(G.sum()), "Porsi_Volume_%": 100 * float(G.sum() / total) if total else 0.0,
            "Rata2_Mingguan": float(G.mean()), "Median_Minggu_Nol": zero_median, "CV_Profil": cv,
            "Perubahan_13mg_%": change,
            "Produk_Representatif": str(codes[result.medoids[pos]]) if pos is not None else "–",
            "Rata2_Jarak_ke_Pusat": float(np.nanmean(result.dist_to_center[mask])) if c != -1 else np.nan,
        })
    return rows


In [ ]:
%%writefile scripts/model_store.py
"""Membangun, menyimpan, dan memuat hasil model klasterisasi (FastDTW · 3 representasi).

Dipakai dua sisi:
* **Notebook Colab** → :func:`build_model_results` lalu :func:`export_results` → ``hasil_model.zip``.
* **Dashboard Streamlit** → :func:`load_results` + :func:`run_to_result` (tanpa menghitung ulang).

Struktur ``model_results.json``::

    {"metadata": {...}, "codes": [...], "W": [[...]],
     "spaces": {rep: {"xy", "explained", "kdist", "linkage": {lk: {"Z", "leaves"}}, "note"}},
     "runs":   {"rep|algo|k|extra": {"labels", "silhouette", "dist_to_center", "centers", "center_kind",
                                     "medoids", "cluster_ids", "center_xy", "center_dm", "membership",
                                     "eps", "notes", "metrics", "n_iter"}},
     "best":   {rep: run_key}}
"""
from __future__ import annotations

import hashlib
import json
import platform
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace
from typing import Callable

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram

from scripts import interactive as ix

ALGORITHMS = ["kmeans", "kmedoids", "agglomerative", "dbscan", "fcm", "fcmdd"]


def run_key(rep: str, algorithm: str, k: int | None = None, extra: str | float | None = None) -> str:
    """Kunci unik satu konfigurasi, mis. ``minmax|agglomerative|3|average`` atau ``zscore|dbscan|0|85``."""
    return f"{rep}|{algorithm}|{0 if algorithm == 'dbscan' else int(k)}|{'' if extra is None else extra}"


def _f(values, digits: int = 4):
    """Array → list angka JSON (NaN → None)."""
    arr = np.asarray(values, dtype=float)
    out = np.round(arr, digits).astype(object)
    out[~np.isfinite(arr)] = None
    return out.tolist()


def _clean(value):
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else round(float(value), 6)
    if isinstance(value, np.integer):
        return int(value)
    return value


def _center_distances(result: ix.ClusterResult, D: np.ndarray) -> np.ndarray:
    """Jarak antar-pusat cluster: medoid → dari matriks FastDTW; DBA → DTW antar-barycenter."""
    k = len(result.cluster_ids)
    if k == 0:
        return np.zeros((0, 0))
    if result.center_kind == "medoid":
        return D[np.ix_(result.medoids, result.medoids)]
    return ix.dtw_batch(np.repeat(result.centers, k, axis=0), np.tile(result.centers, (k, 1)), None).reshape(k, k)


def _pack(result: ix.ClusterResult, D: np.ndarray) -> dict:
    return {
        "labels": result.labels.astype(int).tolist(),
        "silhouette": _f(result.silhouette, 4),
        "dist_to_center": _f(result.dist_to_center, 4),
        "centers": _f(result.centers, 4),
        "center_kind": result.center_kind,
        "medoids": [int(m) for m in result.medoids],
        "cluster_ids": [int(c) for c in result.cluster_ids],
        "center_xy": _f(result.center_xy, 4),
        "center_dm": _f(_center_distances(result, D), 4),
        "membership": _f(result.membership, 4) if result.membership is not None else None,
        "eps": _clean(result.eps),
        "notes": list(result.notes),
        "metrics": {k: _clean(v) for k, v in result.metrics.items()},
        "n_iter": int(result.n_iter),
    }


def select_best(runs: dict, rep: str, n_products: int, cfg) -> tuple[str | None, pd.DataFrame]:
    """Model terbaik satu representasi: lolos guardrail lalu peringkat rata-rata Silhouette↑, DBI↓, CHI↑."""
    rows = []
    for key, run in runs.items():
        if not key.startswith(rep + "|"):
            continue
        m = run["metrics"]
        ok = (m["Cluster"] or 0) >= cfg.MIN_CLUSTERS and (m["Balance"] or 0) >= cfg.MIN_BALANCE \
            and (m["Noise"] or 0) / n_products <= cfg.MAX_NOISE_SHARE and m["Silhouette"] is not None
        rows.append({"Kunci": key, "Lolos_guardrail": ok, **{c: m.get(c) for c in ["Silhouette", "DBI_MDS", "CHI_MDS", "Balance", "Cluster", "Noise", "Eta2_nol", "Waktu_s"]}})
    table = pd.DataFrame(rows)
    passed = table[table["Lolos_guardrail"]].copy()
    if passed.empty:
        return None, table
    ranked = pd.DataFrame(ix.rank_methods(passed.drop(columns="Lolos_guardrail").to_dict("records")))
    return str(ranked.iloc[0]["Kunci"]), ranked


def seed_stability(X: np.ndarray, D: np.ndarray, key: str, cfg, seeds=(42, 52, 62)) -> dict:
    """ARI rata-rata antar-seed untuk model terbaik (hanya metode dengan inisialisasi acak)."""
    from itertools import combinations
    from sklearn.metrics import adjusted_rand_score
    rep, algo, k, extra = key.split("|")
    if algo in ("kmedoids", "agglomerative", "dbscan"):
        return {"ARI": None, "status": "deterministik (tidak bergantung seed)"}
    labels = [ix.run_clustering(X, D, algo, cfg.DISTANCE, k=int(k), fuzziness=cfg.FUZZINESS, seed=s, n_init=1).labels for s in seeds]
    ari = float(np.mean([adjusted_rand_score(a, b) for a, b in combinations(labels, 2)]))
    return {"ARI": round(ari, 4), "status": f"ARI {ari:.3f} " + ("(stabil)" if ari >= 0.75 else "(kurang stabil, < 0,75)")}


def build_model_results(W: np.ndarray, codes: np.ndarray, cfg, cache_dir: str | Path | None = None,
                        log: Callable[[str], None] = print) -> dict:
    """Menjalankan seluruh grid: 3 representasi × FastDTW × 6 algoritma × K/linkage/eps."""
    W = np.asarray(W, dtype=float)
    codes = np.asarray(codes).astype(str)
    started = time.time()
    spaces, runs, best, rankings, stability = {}, {}, {}, {}, {}
    for rep in cfg.REPRESENTATIONS:
        t0 = time.time()
        X = ix.represent(W, rep)
        D, note = ix.distance_matrix(X, cfg.DISTANCE, radius=cfg.FASTDTW_RADIUS, cache_dir=cache_dir)
        emb = ix.classical_mds(D, 5)
        log(f"[{rep}] matriks {ix.DISTANCES[cfg.DISTANCE]} 811×811 siap ({time.time() - t0:.1f} dtk · {note})")
        linkages = {}
        for lk in cfg.LINKAGES:
            res = ix.run_clustering(X, D, "agglomerative", cfg.DISTANCE, k=2, linkage_method=lk)
            linkages[lk] = {"Z": _f(res.linkage, 6), "leaves": dendrogram(res.linkage, no_plot=True, count_sort=True)["leaves"]}
        spaces[rep] = {"xy": _f(emb.coords[:, :2], 4), "explained": _f(emb.explained, 4),
                       "kdist": _f(ix.kdistance(D, cfg.DBSCAN_MIN_SAMPLES), 4), "linkage": linkages, "note": note}
        specs = [(a, k, None, {}) for a in ["kmeans", "kmedoids", "fcm", "fcmdd"] for k in cfg.K_GRID]
        specs += [("agglomerative", k, lk, {"linkage_method": lk}) for lk in cfg.LINKAGES for k in cfg.K_GRID]
        specs += [("dbscan", 0, p, {"min_samples": cfg.DBSCAN_MIN_SAMPLES, "eps_percentile": p}) for p in cfg.DBSCAN_EPS_PERCENTILES]
        for i, (algo, k, extra, kwargs) in enumerate(specs, start=1):
            t1 = time.time()
            res = ix.run_clustering(X, D, algo, cfg.DISTANCE, k=max(k, 2), fuzziness=cfg.FUZZINESS, seed=cfg.RANDOM_STATE,
                                    n_init=cfg.KMEANS_N_INIT, **kwargs)
            res = ix.evaluate(res, D, emb, W)
            key = run_key(rep, algo, k, extra)
            runs[key] = _pack(res, D)
            sil = res.metrics["Silhouette"]
            log(f"  {i:2d}/{len(specs)} {key:32s} silhouette={'–' if not np.isfinite(sil) else f'{sil:.3f}'} "
                f"balance={res.metrics['Balance']:.2f} · {time.time() - t1:.1f} dtk")
        best[rep], rankings[rep] = select_best(runs, rep, len(W), cfg)
        log(f"[{rep}] model terbaik: {best[rep]}")
        if best[rep]:
            stability[rep] = seed_stability(X, D, best[rep], cfg)
            log(f"[{rep}] stabilitas antar-seed: {stability[rep]['status']}")
    metadata = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "n_products": int(W.shape[0]), "n_weeks": int(W.shape[1]), "total_sales": float(W.sum()),
        "data_digest": hashlib.sha256(np.ascontiguousarray(W).tobytes()).hexdigest()[:16],
        "distance": {"name": ix.DISTANCES[cfg.DISTANCE], "radius": cfg.FASTDTW_RADIUS,
                     "notes": {rep: spaces[rep]["note"] for rep in spaces}},
        "config": {k: getattr(cfg, k) for k in ["REPRESENTATIONS", "K_GRID", "LINKAGES", "DBSCAN_MIN_SAMPLES",
                                                 "DBSCAN_EPS_PERCENTILES", "FUZZINESS", "RANDOM_STATE",
                                                 "MIN_BALANCE", "MAX_NOISE_SHARE", "MIN_CLUSTERS"]},
        "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__},
        "runtime_s": round(time.time() - started, 1),
        "stability": stability,
    }
    return {"metadata": metadata, "codes": codes.tolist(), "W": W.astype(int if np.allclose(W, np.round(W)) else float).tolist(),
            "spaces": spaces, "runs": runs, "best": best,
            "_rankings": {rep: table.to_dict("records") for rep, table in rankings.items()}}


def metrics_table(results: dict) -> pd.DataFrame:
    """Tabel metrik semua konfigurasi (satu baris per run)."""
    rows = []
    for key, run in results["runs"].items():
        rep, algo, k, extra = key.split("|")
        m = run["metrics"]
        rows.append({"Representasi": ix.REPRESENTATIONS[rep], "Algoritma": ix.ALGORITHMS[algo], "K": int(k) if algo != "dbscan" else None,
                     "Parameter": extra, **{c: m.get(c) for c in ["Silhouette", "DBI_MDS", "CHI_MDS", "Balance", "Cluster", "Noise", "Eta2_nol", "Waktu_s"]},
                     "Terbaik": key in results["best"].values(), "Kunci": key})
    return pd.DataFrame(rows)


def export_results(results: dict, out_dir: str | Path, zip_path: str | Path | None = None) -> Path:
    """Menulis folder ``hasil_model`` (JSON + CSV pendukung) dan opsional mengemasnya menjadi zip."""
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    payload = {k: v for k, v in results.items() if not k.startswith("_")}
    (out / "model_results.json").write_text(json.dumps(payload, separators=(",", ":"), allow_nan=False), encoding="utf-8")
    (out / "metadata.json").write_text(json.dumps({**results["metadata"], "best": results["best"]}, indent=2, ensure_ascii=False), encoding="utf-8")
    codes = results["codes"]
    weeks = [f"W{i}" for i in range(len(results["W"][0]))]
    pd.DataFrame(results["W"], columns=weeks).assign(Product_Code=codes)[["Product_Code", *weeks]].to_csv(out / "sales.csv", index=False)
    metrics_table(results).to_csv(out / "metrik_semua_model.csv", index=False)
    labels = pd.DataFrame({"Product_Code": codes})
    for rep, key in results["best"].items():
        if key:
            labels[f"{rep}__{key.split('|', 1)[1].replace('|', '_')}"] = results["runs"][key]["labels"]
    labels.to_csv(out / "label_model_terbaik.csv", index=False)
    if zip_path:
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(out.iterdir()):
                archive.write(path, arcname=f"hasil_model/{path.name}")
        return Path(zip_path)
    return out


def load_results(path: str | Path) -> dict:
    """Memuat ``model_results.json`` (folder hasil_model atau path file langsung)."""
    path = Path(path)
    if path.is_dir():
        path = path / "model_results.json"
    return json.loads(path.read_text(encoding="utf-8"))


def run_to_result(run: dict, space: dict, linkage: str | None = None) -> SimpleNamespace:
    """Mengubah satu run tersimpan menjadi objek seperti ``ClusterResult`` untuk visualisasi & profil."""
    arr = lambda v: np.array([np.nan if x is None else x for x in v], dtype=float) if v is not None else None
    centers = np.array([[np.nan if x is None else x for x in row] for row in run["centers"]], dtype=float) if run["centers"] else np.empty((0, 0))
    return SimpleNamespace(
        labels=np.asarray(run["labels"], dtype=int), silhouette=arr(run["silhouette"]), dist_to_center=arr(run["dist_to_center"]),
        centers=centers, center_kind=run["center_kind"], medoids=np.asarray(run["medoids"], dtype=int),
        cluster_ids=np.asarray(run["cluster_ids"], dtype=int),
        center_xy=np.array(run["center_xy"], dtype=float).reshape(-1, 2) if run["center_xy"] else np.empty((0, 2)),
        center_dm=np.array(run["center_dm"], dtype=float) if run["center_dm"] else np.zeros((0, 0)),
        membership=np.array(run["membership"], dtype=float) if run["membership"] is not None else None,
        linkage=np.array(space["linkage"][linkage]["Z"], dtype=float) if linkage else None,
        eps=run["eps"], notes=run["notes"], metrics={k: (np.nan if v is None else v) for k, v in run["metrics"].items()},
        n_iter=run["n_iter"], runtime=run["metrics"].get("Waktu_s") or 0.0,
    )


In [ ]:
%%writefile scripts/visualization.py
"""Seluruh definisi visualisasi Plotly untuk notebook dan dashboard Ruang Sales.

Versi 2: palet kategorikal tervalidasi (warna mengikuti nomor cluster, bukan peringkat),
tanpa grafik sumbu ganda, ditambah visual baru (Pareto, peta panas produk × minggu,
small multiples, peringkat metrik, aliran label antar-metode, grid tuning DBSCAN, dan
keanggotaan fuzzy). Semua nama fungsi lama tetap ada dengan argumen yang kompatibel
sehingga notebook tidak perlu diubah.
"""
from __future__ import annotations

from typing import Iterable

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --------------------------------------------------------------------------------------
# Token tampilan (fallback bila `config` tidak diberikan)
# --------------------------------------------------------------------------------------
_DEFAULTS = {
    "PLOT_PRIMARY": "#00806A",
    "PLOT_ACCENT": "#E06C2B",
    "PLOT_PALETTE": ["#00806A", "#E06C2B", "#3A6FD9", "#C8457F", "#7B5BD6"],
    "PLOT_NOISE": "#9AA39E",
    "PLOT_INK": "#13201B",
    "PLOT_MUTED": "#6B7770",
    "PLOT_GRID": "#EEEBE2",
    "PLOT_SEQUENTIAL": ["#F5F3EC", "#BFE3D6", "#55B196", "#00806A", "#00483A"],
    "PLOT_SEQUENTIAL_WARM": ["#FBEFE7", "#F6CFB6", "#EDA47C", "#E06C2B", "#A9481A"],
    "PLOT_FONT": "Plus Jakarta Sans, Arial",
}
WEEK_TICKS = [0, 8, 16, 24, 32, 40, 51]


def _cfg(config: object | None, name: str):
    """Mengambil token tampilan dari config, atau nilai bawaan versi 2."""
    return getattr(config, name, _DEFAULTS[name]) if config is not None else _DEFAULTS[name]


def cluster_color(cluster_id: int, config: object | None = None) -> str:
    """Warna tetap untuk satu nomor cluster; noise (−1) selalu abu-abu netral."""
    if int(cluster_id) == -1:
        return _cfg(config, "PLOT_NOISE")
    palette = _cfg(config, "PLOT_PALETTE")
    return palette[int(cluster_id) % len(palette)]


def cluster_name(cluster_id: int) -> str:
    """Label pendek cluster untuk legenda dan tooltip."""
    return "Noise" if int(cluster_id) == -1 else f"C{int(cluster_id)}"


def _scale(colors: list[str]) -> list[list]:
    """Mengubah daftar warna menjadi colorscale Plotly berjarak sama."""
    n = len(colors) - 1
    return [[i / n, color] for i, color in enumerate(colors)]


def _hex_to_rgba(color: str, alpha: float = 0.12) -> str:
    """Mengubah warna HEX menjadi RGBA untuk pita rentang antar-persentil."""
    value = color.lstrip("#")
    red, green, blue = (int(value[index:index + 2], 16) for index in (0, 2, 4))
    return f"rgba({red},{green},{blue},{alpha})"


def style_figure(fig: go.Figure, title: str | None = None, height: int = 360, config: object | None = None) -> go.Figure:
    """Menerapkan gaya grafik Ruang Sales v2: latar transparan, grid tipis, legend horizontal."""
    ink, muted, grid = _cfg(config, "PLOT_INK"), _cfg(config, "PLOT_MUTED"), _cfg(config, "PLOT_GRID")
    fig.update_layout(
        template="none",
        title=dict(text=title, font=dict(size=14, color=ink)) if title else None,
        height=height,
        margin=dict(l=8, r=24, t=56 if title else 24, b=12),
        font=dict(family=_cfg(config, "PLOT_FONT"), size=11.5, color=muted),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        colorway=_cfg(config, "PLOT_PALETTE"),
        legend=dict(orientation="h", y=1.08, x=0, yanchor="bottom", font=dict(size=11.5, color=ink), bgcolor="rgba(0,0,0,0)"),
        hoverlabel=dict(bgcolor="#13201B", bordercolor="#13201B", font=dict(size=12, color="#F4F7F5", family=_cfg(config, "PLOT_FONT"))),
        separators=",.",
        bargap=0.25,
    )
    fig.update_xaxes(showgrid=False, automargin=True, zeroline=False, linecolor=grid, tickcolor=grid)
    fig.update_yaxes(gridcolor=grid, gridwidth=1, zeroline=False, automargin=True)
    return fig


def _weekly_axis(fig: go.Figure, ylabel: str, row: int | None = None, col: int | None = None) -> go.Figure:
    """Merapikan sumbu minggu agar label W0–W51 mudah dibaca."""
    kwargs = dict(row=row, col=col) if row else {}
    fig.update_xaxes(title_text="Minggu", tickmode="array", tickvals=WEEK_TICKS,
                     ticktext=[f"W{t}" for t in WEEK_TICKS], **kwargs)
    fig.update_yaxes(title_text=ylabel, **kwargs)
    return fig


def _rolling(values: Iterable[float], window: int) -> np.ndarray:
    """Rata-rata bergerak sederhana; titik awal yang belum lengkap diisi NaN."""
    return pd.Series(np.asarray(list(values), dtype=float)).rolling(window).mean().to_numpy()


# --------------------------------------------------------------------------------------
# 01 · Ringkasan
# --------------------------------------------------------------------------------------
def create_weekly_sales_figure(weekly_stats: pd.DataFrame, config: object | None = None, rolling: int = 4) -> go.Figure:
    """Tren total volume mingguan + rata-rata bergerak dan penanda puncak/terendah."""
    primary, ink = _cfg(config, "PLOT_PRIMARY"), _cfg(config, "PLOT_INK")
    x, y = weekly_stats["Minggu"], weekly_stats["Total_Penjualan"]
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x, y=y, mode="lines", line=dict(width=2.4, color=primary), fill="tozeroy",
        fillcolor=_hex_to_rgba(primary, 0.10), name="Total penjualan",
        hovertemplate="W%{x}<br>Volume: %{y:,.0f}<extra></extra>",
    ))
    if rolling and rolling > 1:
        fig.add_trace(go.Scatter(
            x=x, y=_rolling(y, rolling), mode="lines", name=f"Rata-rata bergerak {rolling} minggu",
            line=dict(width=1.6, color=ink, dash="dash"), hovertemplate="W%{x}<br>MA: %{y:,.0f}<extra></extra>",
        ))
    max_idx, min_idx = int(y.idxmax()), int(y.idxmin())
    fig.add_trace(go.Scatter(
        x=x.loc[[max_idx, min_idx]], y=y.loc[[max_idx, min_idx]], mode="markers+text",
        text=["Puncak", "Terendah"], textposition=["top center", "bottom right"],
        textfont=dict(color=ink, size=11), showlegend=False,
        marker=dict(size=11, color=[_cfg(config, "PLOT_ACCENT"), _cfg(config, "PLOT_PALETTE")[3]], line=dict(color="#fff", width=2)),
        hovertemplate="W%{x}<br>Volume: %{y:,.0f}<extra></extra>",
    ))
    _weekly_axis(fig, "Volume penjualan asli")
    fig.update_yaxes(rangemode="tozero")
    fig.update_layout(hovermode="x unified")
    return style_figure(fig, None, 340, config)


def create_active_products_figure(weekly_stats: pd.DataFrame, config: object | None = None) -> go.Figure:
    """Persentase produk aktif per minggu — grafik terpisah, bukan sumbu kedua."""
    color = _cfg(config, "PLOT_PALETTE")[4]
    fig = go.Figure(go.Scatter(
        x=weekly_stats["Minggu"], y=weekly_stats["Persen_Produk_Aktif"], mode="lines",
        line=dict(width=2, color=color), fill="tozeroy", fillcolor=_hex_to_rgba(color, 0.10),
        name="Produk aktif", hovertemplate="W%{x}<br>Produk aktif: %{y:.1f}%<extra></extra>",
    ))
    _weekly_axis(fig, "Produk aktif (%)")
    fig.update_yaxes(range=[0, 100], ticksuffix="%")
    return style_figure(fig, None, 190, config)


def create_pareto_figure(product_stats: pd.DataFrame, config: object | None = None) -> go.Figure:
    """Kurva konsentrasi volume: % produk (urut terbesar) vs % kumulatif volume."""
    primary, accent = _cfg(config, "PLOT_PRIMARY"), _cfg(config, "PLOT_ACCENT")
    totals = np.sort(product_stats["Total volume"].to_numpy(dtype=float))[::-1]
    cumulative = 100 * np.cumsum(totals) / totals.sum()
    share = 100 * np.arange(1, len(totals) + 1) / len(totals)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode="lines", line=dict(color="#C9C3B3", dash="dot", width=1),
                             hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(
        x=np.r_[0, share], y=np.r_[0, cumulative], mode="lines", line=dict(color=primary, width=2.4),
        fill="tozeroy", fillcolor=_hex_to_rgba(primary, 0.10), name="Kumulatif volume",
        hovertemplate="%{x:.1f}% produk teratas<br>= %{y:.1f}% volume<extra></extra>",
    ))
    top20 = float(np.interp(20, share, cumulative))
    p80 = float(share[np.searchsorted(cumulative, 80)])
    fig.add_trace(go.Scatter(
        x=[20, p80], y=[top20, 80], mode="markers+text", showlegend=False,
        text=[f"20% produk → {top20:.1f}% volume", f"80% volume ← {p80:.1f}% produk"],
        textposition="bottom right", textfont=dict(color=_cfg(config, "PLOT_INK"), size=11),
        marker=dict(size=10, color=accent, line=dict(color="#fff", width=2)), hoverinfo="skip",
    ))
    fig.update_xaxes(title_text="% produk (urut volume terbesar)", range=[0, 100], ticksuffix="%")
    fig.update_yaxes(title_text="% kumulatif volume", range=[0, 102], ticksuffix="%")
    return style_figure(fig, None, 380, config)


def create_top_products_figure(product_stats: pd.DataFrame, top_n: int = 10, config: object | None = None,
                               cluster_by_product: dict | None = None) -> go.Figure:
    """Bar horizontal produk dengan volume tertinggi; diwarnai cluster bila tersedia."""
    top = product_stats.nlargest(int(top_n), "Total volume").sort_values("Total volume")
    if cluster_by_product:
        colors = [cluster_color(cluster_by_product.get(code, 0), config) for code in top["Produk"]]
        clusters = [cluster_name(cluster_by_product.get(code, 0)) for code in top["Produk"]]
    else:
        colors, clusters = _cfg(config, "PLOT_PRIMARY"), ["–"] * len(top)
    fig = go.Figure(go.Bar(
        x=top["Total volume"], y=top["Produk"], orientation="h", marker=dict(color=colors, cornerradius=4),
        text=top["Total volume"], texttemplate="%{text:,.0f}", textposition="outside", cliponaxis=False,
        customdata=np.c_[clusters, top["Minggu nol"], top["Minggu puncak"]],
        hovertemplate="<b>%{y}</b><br>Total: %{x:,.0f}<br>Cluster: %{customdata[0]}<br>Minggu nol: %{customdata[1]}<br>Puncak: %{customdata[2]}<extra></extra>",
    ))
    fig.update_xaxes(title_text="Total volume · 52 minggu", showgrid=True, gridcolor=_cfg(config, "PLOT_GRID"))
    fig.update_yaxes(title_text=None, showgrid=False)
    return style_figure(fig, None, 390, config)


def create_zero_weeks_figure(product_stats: pd.DataFrame, config: object | None = None) -> go.Figure:
    """Distribusi banyaknya minggu tanpa penjualan pada setiap produk (0–52)."""
    counts = product_stats["Minggu nol"].value_counts().sort_index().reindex(range(53), fill_value=0)
    fig = go.Figure(go.Bar(
        x=counts.index, y=counts.values, marker=dict(color=_cfg(config, "PLOT_ACCENT"), cornerradius=2),
        hovertemplate="%{x} minggu nol<br>%{y} produk<extra></extra>",
    ))
    fig.update_xaxes(title_text="Jumlah minggu tanpa penjualan", dtick=5, range=[-0.6, 52.6])
    fig.update_yaxes(title_text="Jumlah produk")
    fig.update_layout(bargap=0.12)
    return style_figure(fig, None, 390, config)


# --------------------------------------------------------------------------------------
# 02 · Eksplorasi
# --------------------------------------------------------------------------------------
def create_exploration_figures(
    product_stats: pd.DataFrame,
    W_original: np.ndarray,
    selected_product: str,
    config: object | None = None,
    labels: np.ndarray | None = None,
    compare_product: str | None = None,
    scale: str = "raw",
    X_scaled: np.ndarray | None = None,
) -> dict[str, go.Figure]:
    """Histogram log, scatter aktivitas–volume (warna cluster), dan detail deret produk."""
    values = np.asarray(W_original, dtype=float)
    codes = product_stats["Produk"].astype(str).to_numpy()
    if selected_product not in set(codes):
        selected_product = str(codes[0])
    position = int(np.flatnonzero(codes == selected_product)[0])
    y_labels = np.zeros(len(codes), dtype=int) if labels is None else np.asarray(labels, dtype=int)

    # Histogram dengan bin logaritmik (distribusi total sangat menceng).
    edges = [0, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000, np.inf]
    names = ["<10", "10+", "20+", "50+", "100+", "200+", "500+", "1rb+", "2rb+", "5rb+"]
    totals = product_stats["Total volume"].to_numpy(dtype=float)
    counts = [int(((totals >= lo) & (totals < hi)).sum()) for lo, hi in zip(edges[:-1], edges[1:])]
    hist = go.Figure(go.Bar(x=names, y=counts, marker=dict(color=_cfg(config, "PLOT_PALETTE")[2], cornerradius=4),
                            hovertemplate="Total %{x}<br>%{y} produk<extra></extra>"))
    hist.update_xaxes(title_text="Total volume per produk · 52 minggu (bin log)")
    hist.update_yaxes(title_text="Jumlah produk")
    hist = style_figure(hist, None, 360, config)

    scatter = go.Figure()
    for cluster_id in sorted(np.unique(y_labels)):
        mask = y_labels == cluster_id
        frame = product_stats.loc[mask]
        scatter.add_trace(go.Scattergl(
            x=frame["Minggu aktif (%)"], y=frame["Total volume"].clip(lower=1), mode="markers",
            name=cluster_name(cluster_id), customdata=np.c_[frame["Produk"], frame["Minggu nol"]],
            marker=dict(size=7, color=cluster_color(cluster_id, config), opacity=0.6, line=dict(color="#fff", width=0.6)),
            hovertemplate="<b>%{customdata[0]}</b><br>Aktif: %{x:.1f}%<br>Total: %{y:,.0f}<br>Minggu nol: %{customdata[1]}<extra></extra>",
        ))
    row = product_stats.iloc[position]
    scatter.add_trace(go.Scatter(
        x=[row["Minggu aktif (%)"]], y=[max(1.0, row["Total volume"])], mode="markers+text", text=[selected_product],
        textposition="top center", showlegend=False, hoverinfo="skip",
        marker=dict(size=14, color="rgba(0,0,0,0)", line=dict(color=_cfg(config, "PLOT_INK"), width=2.5)),
    ))
    scatter.update_xaxes(title_text="Minggu aktif (%)", ticksuffix="%")
    scatter.update_yaxes(title_text="Total volume (skala log)", type="log")
    scatter = style_figure(scatter, None, 380, config)

    source = values if scale == "raw" or X_scaled is None else np.asarray(X_scaled, dtype=float)
    weeks = np.arange(values.shape[1])
    detail = go.Figure()
    primary = _cfg(config, "PLOT_PRIMARY")
    detail.add_trace(go.Scatter(
        x=weeks, y=source[position], mode="lines+markers", name=selected_product,
        line=dict(width=2.4, color=primary), marker=dict(size=4), fill="tozeroy" if not compare_product else None,
        fillcolor=_hex_to_rgba(primary, 0.08), hovertemplate="W%{x}: %{y:,.2f}<extra>" + selected_product + "</extra>",
    ))
    if compare_product and compare_product in set(codes):
        other = int(np.flatnonzero(codes == compare_product)[0])
        detail.add_trace(go.Scatter(x=weeks, y=source[other], mode="lines", name=compare_product,
                                    line=dict(width=2.2, color=_cfg(config, "PLOT_ACCENT")),
                                    hovertemplate="W%{x}: %{y:,.2f}<extra>" + compare_product + "</extra>"))
    else:
        detail.add_trace(go.Scatter(x=weeks, y=_rolling(source[position], 4), mode="lines", name="Rata-rata bergerak 4 minggu",
                                    line=dict(width=1.5, color=_cfg(config, "PLOT_INK"), dash="dash"), hoverinfo="skip"))
    own_cluster = y_labels[position]
    median = np.median(source[y_labels == own_cluster], axis=0)
    detail.add_trace(go.Scatter(x=weeks, y=median, mode="lines", name=f"Median {cluster_name(own_cluster)}",
                                line=dict(width=1.4, color=_cfg(config, "PLOT_PALETTE")[2], dash="dot"),
                                hovertemplate="W%{x}: %{y:,.2f}<extra>median cluster</extra>"))
    _weekly_axis(detail, {"raw": "Penjualan asli", "zscore": "Nilai z-score"}.get(scale, "Nilai Min-Max"))
    detail.update_yaxes(rangemode="tozero")
    detail.update_layout(hovermode="x unified")
    detail = style_figure(detail, None, 360, config)
    return {"sales_distribution": hist, "activity_vs_volume": scatter, "product_detail": detail}


def create_sales_heatmap_figure(
    W_original: np.ndarray,
    X_scaled: np.ndarray,
    labels: np.ndarray,
    product_codes: np.ndarray,
    mode: str = "minmax",
    config: object | None = None,
) -> go.Figure:
    """Peta panas produk × minggu, dikelompokkan per cluster lalu diurutkan dari volume terbesar."""
    W = np.asarray(W_original, dtype=float)
    X = np.asarray(X_scaled, dtype=float)
    y = np.asarray(labels, dtype=int)
    codes = np.asarray(product_codes).astype(str)
    order = np.lexsort((-W.sum(axis=1), y))
    if mode == "zscore":  # Scale: nilai z per produk; nol tetap berwarna karena z punya arti di bawah rata-rata
        z = np.clip(X[order], -2.5, 2.5)
    else:
        z = X[order] if mode == "minmax" else np.log10(1 + W[order])
        z = np.where(W[order] == 0, np.nan, z)  # nol = putih
    text = np.array([[f"{codes[i]} · {cluster_name(y[i])}"] * W.shape[1] for i in order])
    fig = go.Figure(go.Heatmap(
        z=z, x=[f"W{i}" for i in range(W.shape[1])], y=np.arange(len(order)), customdata=W[order], text=text,
        colorscale=_scale(_cfg(config, "PLOT_SEQUENTIAL")), hoverongaps=False, showscale=True,
        colorbar=dict(title=dict(text={"minmax": "Min-Max", "zscore": "z-score"}.get(mode, "log₁₀(1+unit)"), side="right"), thickness=10, len=0.6),
        hovertemplate="%{text}<br>%{x}: %{customdata:,.0f} unit<br>nilai warna %{z:.2f}<extra></extra>",
    ))
    boundaries = np.flatnonzero(np.diff(y[order])) + 0.5
    for b in boundaries:
        fig.add_hline(y=b, line=dict(color=_cfg(config, "PLOT_INK"), width=1.5))
    start = 0
    for cluster_id in sorted(np.unique(y)):
        n = int((y == cluster_id).sum())
        fig.add_annotation(x=-0.01, xref="paper", y=start + n / 2, yref="y", xanchor="right", showarrow=False,
                           text=f"<b>{cluster_name(cluster_id)}</b><br>{n}", font=dict(size=11, color=cluster_color(cluster_id, config)))
        start += n
    fig.update_yaxes(autorange="reversed", showticklabels=False, showgrid=False, title_text=None)
    fig.update_xaxes(tickmode="array", tickvals=[f"W{t}" for t in WEEK_TICKS], showgrid=False)
    fig = style_figure(fig, None, 560, config)
    fig.update_layout(margin=dict(l=60, r=10, t=10, b=10))
    return fig


# --------------------------------------------------------------------------------------
# 03 · Segmentasi
# --------------------------------------------------------------------------------------
def create_cluster_pattern_figure(
    W_or_X: np.ndarray,
    labels: np.ndarray,
    scale_label: str,
    config: object | None = None,
    stat: str = "median",
    band: tuple[float, float] = (10, 90),
    names: dict | None = None,
) -> go.Figure:
    """Profil mingguan tiap cluster: garis median/rata-rata dengan pita persentil."""
    values = np.asarray(W_or_X, dtype=float)
    y = np.asarray(labels, dtype=int)
    weeks = np.arange(values.shape[1])
    fig = go.Figure()
    for cluster_id in sorted(np.unique(y)):
        group = values[y == cluster_id]
        center = group.mean(axis=0) if stat == "mean" else np.median(group, axis=0)
        low, high = np.percentile(group, band[0], axis=0), np.percentile(group, band[1], axis=0)
        color = cluster_color(cluster_id, config)
        label = f"{cluster_name(cluster_id)}" + (f" · {names[cluster_id]}" if names and cluster_id in names else "") + f" (n={len(group)})"
        fig.add_trace(go.Scatter(x=weeks, y=high, mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip", legendgroup=label))
        fig.add_trace(go.Scatter(x=weeks, y=low, mode="lines", line=dict(width=0), fill="tonexty",
                                 fillcolor=_hex_to_rgba(color, 0.13), showlegend=False, hoverinfo="skip", legendgroup=label))
        fig.add_trace(go.Scatter(
            x=weeks, y=center, mode="lines", legendgroup=label, name=label,
            line=dict(width=2.8, color=color, dash="dash" if cluster_id == -1 else "solid"),
            hovertemplate="%{y:.3f}<extra>" + cluster_name(cluster_id) + "</extra>",
        ))
    _weekly_axis(fig, scale_label)
    fig.update_yaxes(rangemode="tozero")
    fig.update_layout(hovermode="x unified")
    return style_figure(fig, None, 400, config)


def create_cluster_small_multiples_figure(
    X_scaled: np.ndarray,
    labels: np.ndarray,
    product_codes: np.ndarray,
    medoids: dict | None = None,
    config: object | None = None,
    sample: int = 40,
    seed: int = 42,
) -> go.Figure:
    """Satu panel per cluster: sampel anggota (garis tipis), rata-rata tebal, dan medoid putus-putus."""
    X = np.asarray(X_scaled, dtype=float)
    y = np.asarray(labels, dtype=int)
    codes = np.asarray(product_codes).astype(str)
    clusters = sorted(np.unique(y))
    rng = np.random.default_rng(seed)
    fig = make_subplots(rows=1, cols=len(clusters), shared_yaxes=True, horizontal_spacing=0.04,
                        subplot_titles=[f"{cluster_name(c)} · {(y == c).sum()} produk" for c in clusters])
    weeks = np.arange(X.shape[1])
    for col, cluster_id in enumerate(clusters, start=1):
        idx = np.flatnonzero(y == cluster_id)
        pick = rng.choice(idx, size=min(sample, len(idx)), replace=False)
        color = cluster_color(cluster_id, config)
        for i in pick:
            fig.add_trace(go.Scattergl(x=weeks, y=X[i], mode="lines", line=dict(width=0.8, color=_hex_to_rgba(color, 0.25)),
                                       showlegend=False, hovertemplate=codes[i] + " · W%{x}: %{y:.2f}<extra></extra>"), row=1, col=col)
        fig.add_trace(go.Scatter(x=weeks, y=X[idx].mean(axis=0), mode="lines", line=dict(width=2.6, color=_cfg(config, "PLOT_INK")),
                                 name="Rata-rata", showlegend=col == 1, legendgroup="mean",
                                 hovertemplate="Rata-rata W%{x}: %{y:.3f}<extra></extra>"), row=1, col=col)
        medoid = (medoids or {}).get(cluster_id)
        if medoid in set(codes):
            m_idx = int(np.flatnonzero(codes == medoid)[0])
            fig.add_trace(go.Scatter(x=weeks, y=X[m_idx], mode="lines", line=dict(width=1.8, color=color, dash="dash"),
                                     name=f"Medoid", showlegend=col == 1, legendgroup="medoid",
                                     hovertemplate=f"Medoid {medoid} · W%{{x}}: %{{y:.2f}}<extra></extra>"), row=1, col=col)
        _weekly_axis(fig, "Min-Max" if col == 1 else "", row=1, col=col)
        fig.update_xaxes(title_text=None, row=1, col=col)
    fig.update_yaxes(range=[0, 1.02])
    fig = style_figure(fig, None, 300, config)
    fig.update_annotations(font=dict(size=12, color=_cfg(config, "PLOT_INK")))
    return fig


def create_cluster_contribution_figure(profiles: pd.DataFrame, config: object | None = None) -> go.Figure:
    """Membandingkan proporsi jumlah produk dan proporsi volume setiap cluster."""
    labels = [cluster_name(value) for value in profiles["Cluster"]]
    palette = _cfg(config, "PLOT_PALETTE")
    fig = go.Figure()
    fig.add_bar(y=labels, x=profiles["Persen_Produk"], orientation="h", name="Porsi produk",
                marker=dict(color=palette[2], cornerradius=4), text=profiles["Persen_Produk"], texttemplate="%{text:.1f}%",
                textposition="outside", cliponaxis=False, hovertemplate="%{y}: %{x:.1f}% produk<extra></extra>")
    fig.add_bar(y=labels, x=profiles["Porsi_Volume"], orientation="h", name="Porsi volume",
                marker=dict(color=palette[4], cornerradius=4), text=profiles["Porsi_Volume"], texttemplate="%{text:.1f}%",
                textposition="outside", cliponaxis=False, hovertemplate="%{y}: %{x:.1f}% volume<extra></extra>")
    fig.update_layout(barmode="group", bargap=0.35, bargroupgap=0.08)
    fig.update_xaxes(title_text="Persentase dari keseluruhan", ticksuffix="%", range=[0, 118], showgrid=True, gridcolor=_cfg(config, "PLOT_GRID"))
    fig.update_yaxes(title_text=None, autorange="reversed", showgrid=False)
    return style_figure(fig, None, 300, config)


def create_membership_figure(max_membership: np.ndarray, labels: np.ndarray, config: object | None = None) -> go.Figure:
    """Histogram derajat keanggotaan fuzzy tertinggi per produk, dipisah per cluster."""
    u = np.asarray(max_membership, dtype=float)
    y = np.asarray(labels, dtype=int)
    fig = go.Figure()
    for cluster_id in sorted(np.unique(y)):
        fig.add_trace(go.Histogram(x=u[y == cluster_id], name=cluster_name(cluster_id), xbins=dict(start=0.5, end=1.0, size=0.025),
                                   marker=dict(color=cluster_color(cluster_id, config)), opacity=0.85,
                                   hovertemplate="u %{x}<br>%{y} produk<extra>" + cluster_name(cluster_id) + "</extra>"))
    fig.add_vline(x=0.6, line=dict(color=_cfg(config, "PLOT_MUTED"), width=1, dash="dot"),
                  annotation_text="perbatasan", annotation_position="top right")
    fig.update_layout(barmode="group")
    fig.update_xaxes(title_text="Derajat keanggotaan maksimum", range=[0.48, 1.0])
    fig.update_yaxes(title_text="Jumlah produk")
    return style_figure(fig, None, 340, config)


# --------------------------------------------------------------------------------------
# 04 · Perbandingan
# --------------------------------------------------------------------------------------
METRIC_SPECS = [
    ("Silhouette_FastDTW", "Silhouette ↑", True),
    ("DBI_Medoid_FastDTW", "DBI medoid ↓", False),
    ("CHI_MinMax_Euclidean", "CHI ↑", True),
    ("DBI_MinMax_Euclidean", "DBI Eucl. ↓", False),
    ("Coverage", "Coverage ↑", True),
    ("Eta2_Minggu_Nol", "η² nol ↓", False),
    ("Stabilitas_ARI", "Stab. ARI ↑", True),
]


def metric_ranks(metrics: pd.DataFrame) -> pd.DataFrame:
    """Peringkat per metrik (1 = terbaik) dengan arah ↑/↓ dan penanganan nilai seri."""
    ranks = pd.DataFrame(index=metrics["Metode"])
    for column, _, higher in METRIC_SPECS:
        series = metrics.set_index("Metode")[column].astype(float)
        ranks[column] = series.rank(ascending=not higher, method="min")
    return ranks


def create_metrics_comparison_figure(metrics: pd.DataFrame, config: object | None = None, highlight: str | None = None) -> go.Figure:
    """Empat indeks evaluasi utama dalam small multiples; satu metode dapat disorot."""
    highlight = highlight or getattr(config, "DEFAULT_METHOD", None)
    specs = METRIC_SPECS[:2] + [METRIC_SPECS[2], METRIC_SPECS[6]]
    fig = make_subplots(rows=2, cols=2, subplot_titles=[s[1] for s in specs], horizontal_spacing=0.18, vertical_spacing=0.2)
    for index, (metric, _, higher) in enumerate(specs):
        row, col = index // 2 + 1, index % 2 + 1
        ordered = metrics.sort_values(metric, ascending=not higher, na_position="last")
        fig.add_trace(go.Bar(
            x=ordered[metric], y=ordered["Metode"], orientation="h", showlegend=False,
            marker=dict(color=[_cfg(config, "PLOT_ACCENT") if m == highlight else _cfg(config, "PLOT_PRIMARY") for m in ordered["Metode"]], cornerradius=3),
            text=ordered[metric].map(lambda v: "tidak diuji" if pd.isna(v) else f"{v:.3f}"),
            textposition="outside", cliponaxis=False, hovertemplate="%{y}<br>%{x:.4f}<extra></extra>",
        ), row=row, col=col)
        fig.update_yaxes(autorange="reversed", showgrid=False, row=row, col=col)
        fig.update_xaxes(showgrid=True, gridcolor=_cfg(config, "PLOT_GRID"), row=row, col=col)
    fig = style_figure(fig, None, 620, config)
    fig.update_annotations(font=dict(size=12, color=_cfg(config, "PLOT_INK")))
    return fig


def create_rank_heatmap_figure(metrics: pd.DataFrame, config: object | None = None, display_names: dict | None = None) -> go.Figure:
    """Peta peringkat metode × metrik (1 gelap = terbaik), sudah memperhitungkan arah ↑/↓."""
    ranks = metric_ranks(metrics)
    names = [display_names.get(m, m) if display_names else m for m in ranks.index]
    raw = metrics.set_index("Metode")[[c for c, _, _ in METRIC_SPECS]]
    text = ranks.map(lambda v: "–" if pd.isna(v) else f"#{int(v)}")
    fig = go.Figure(go.Heatmap(
        z=ranks.to_numpy(), x=[label for _, label, _ in METRIC_SPECS], y=names, text=text.to_numpy(), texttemplate="%{text}",
        customdata=raw.to_numpy(), zmin=1, zmax=len(ranks), colorscale=_scale(_cfg(config, "PLOT_SEQUENTIAL")[::-1]),
        showscale=False, xgap=3, ygap=3, hoverongaps=False, textfont=dict(size=12),
        hovertemplate="%{y} · %{x}<br>Peringkat %{text} · nilai %{customdata:.3f}<extra></extra>",
    ))
    fig.update_yaxes(autorange="reversed", showgrid=False)
    fig.update_xaxes(side="top", showgrid=False)
    return style_figure(fig, None, 340, config)


def create_coverage_figure(metrics: pd.DataFrame, config: object | None = None, labels_frame: pd.DataFrame | None = None) -> go.Figure:
    """Komposisi produk per cluster untuk tiap metode (stacked 100%). Tanpa sumbu ganda.

    Jika `labels_frame` tidak diberikan (pemanggilan lama di notebook), grafik menampilkan
    coverage saja dengan jumlah cluster sebagai label teks.
    """
    methods = metrics["Metode"].tolist()
    fig = go.Figure()
    if labels_frame is None:
        fig.add_bar(y=methods, x=100 * metrics["Coverage"], orientation="h", marker=dict(color=_cfg(config, "PLOT_PRIMARY"), cornerradius=4),
                    text=[f"{100 * c:.1f}% · {k} cluster" for c, k in zip(metrics["Coverage"], metrics["Jumlah_Cluster"])],
                    textposition="inside", hovertemplate="%{y}: %{x:.1f}%<extra></extra>", name="Coverage")
    else:
        all_clusters = sorted({int(c) for m in methods for c in np.unique(labels_frame[m])}, key=lambda c: (c == -1, c))
        for cluster_id in all_clusters:
            shares = [100 * float((labels_frame[m].to_numpy(dtype=int) == cluster_id).mean()) for m in methods]
            fig.add_bar(y=methods, x=shares, orientation="h", name=cluster_name(cluster_id), marker=dict(color=cluster_color(cluster_id, config)),
                        text=[f"{s:.0f}%" if s >= 6 else "" for s in shares], textposition="inside", insidetextanchor="middle",
                        textfont=dict(color="#fff"), hovertemplate="%{y} · " + cluster_name(cluster_id) + ": %{x:.1f}%<extra></extra>")
        fig.update_layout(barmode="stack")
    fig.update_xaxes(title_text="Porsi produk (%)", range=[0, 100], ticksuffix="%", showgrid=True, gridcolor=_cfg(config, "PLOT_GRID"))
    fig.update_yaxes(autorange="reversed", showgrid=False, title_text=None)
    fig.update_layout(bargap=0.35)
    return style_figure(fig, None, 340, config)


def create_similarity_heatmap(matrix: pd.DataFrame, metric_name: str = "ARI", config: object | None = None) -> go.Figure:
    """Heatmap kesepakatan partisi berdasarkan ARI atau NMI (ramp satu hue)."""
    fig = go.Figure(go.Heatmap(
        z=matrix.to_numpy(), x=matrix.columns.tolist(), y=matrix.index.tolist(), zmin=0, zmax=1,
        colorscale=_scale(_cfg(config, "PLOT_SEQUENTIAL")), text=matrix.round(2).to_numpy(), texttemplate="%{text:.2f}",
        xgap=3, ygap=3, colorbar=dict(title=dict(text=metric_name), thickness=10, len=0.7),
        hovertemplate="%{y} × %{x}<br>" + metric_name + " = %{z:.3f}<extra></extra>",
    ))
    fig.update_yaxes(autorange="reversed", showgrid=False)
    fig.update_xaxes(showgrid=False, tickangle=-25)
    return style_figure(fig, None, 420, config)


def create_label_flow_sankey(labels_a: np.ndarray, labels_b: np.ndarray, name_a: str, name_b: str, config: object | None = None) -> go.Figure:
    """Diagram aliran: ke cluster mana produk dari metode A berpindah pada metode B."""
    a, b = np.asarray(labels_a, dtype=int), np.asarray(labels_b, dtype=int)
    left, right = sorted(np.unique(a)), sorted(np.unique(b))
    nodes = [f"{name_a} · {cluster_name(c)}" for c in left] + [f"{name_b} · {cluster_name(c)}" for c in right]
    colors = [cluster_color(c, config) for c in left] + [cluster_color(c, config) for c in right]
    source, target, value, link_colors = [], [], [], []
    for i, ca in enumerate(left):
        for j, cb in enumerate(right):
            n = int(((a == ca) & (b == cb)).sum())
            if n:
                source.append(i); target.append(len(left) + j); value.append(n)
                link_colors.append(_hex_to_rgba(cluster_color(ca, config), 0.32))
    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(label=nodes, color=colors, pad=18, thickness=14, line=dict(color="#fff", width=0.5),
                  hovertemplate="%{label}<br>%{value} produk<extra></extra>"),
        link=dict(source=source, target=target, value=value, color=link_colors,
                  hovertemplate="%{source.label} → %{target.label}<br>%{value} produk<extra></extra>"),
    ))
    return style_figure(fig, None, 340, config)


def create_tuning_k_figure(tuning_per_k: pd.DataFrame, config: object | None = None) -> go.Figure:
    """Perubahan Silhouette dan DBI terhadap jumlah cluster aktual."""
    fig = make_subplots(rows=1, cols=2, subplot_titles=["Silhouette FastDTW ↑", "DBI medoid FastDTW ↓"])
    palette = _cfg(config, "PLOT_PALETTE")
    for index, (method, subset) in enumerate(tuning_per_k.groupby("Metode")):
        subset = subset.sort_values("k_Aktual")
        color = palette[index % len(palette)]
        fig.add_trace(go.Scatter(x=subset["k_Aktual"], y=subset["Silhouette_FastDTW"], mode="lines+markers", name=method,
                                 legendgroup=method, line=dict(color=color, width=2), marker=dict(size=9)), row=1, col=1)
        fig.add_trace(go.Scatter(x=subset["k_Aktual"], y=subset["DBI_Medoid_FastDTW"], mode="lines+markers", name=method,
                                 legendgroup=method, showlegend=False, line=dict(color=color, width=2), marker=dict(size=9)), row=1, col=2)
    fig.update_xaxes(title_text="Jumlah cluster aktual (k)", dtick=1)
    fig = style_figure(fig, None, 420, config)
    fig.update_annotations(font=dict(size=12, color=_cfg(config, "PLOT_INK")))
    return fig


def create_dbscan_tuning_figure(dbscan_tuning: pd.DataFrame, config: object | None = None) -> go.Figure:
    """Trade-off Silhouette non-noise dan proporsi noise terhadap eps DBSCAN (garis)."""
    fig = make_subplots(rows=1, cols=2, subplot_titles=["Silhouette non-noise vs eps", "Produk noise (%) vs eps"])
    palette = _cfg(config, "PLOT_PALETTE")
    for index, (min_samples, subset) in enumerate(dbscan_tuning.groupby("min_samples")):
        subset = subset.sort_values("eps")
        color = palette[index % len(palette)]
        fig.add_trace(go.Scatter(x=subset["eps"], y=subset["Silhouette_FastDTW"], mode="lines+markers", name=f"min_samples={int(min_samples)}",
                                 legendgroup=str(min_samples), line=dict(color=color, width=2)), row=1, col=1)
        fig.add_trace(go.Scatter(x=subset["eps"], y=100 * (1 - subset["Coverage"]), mode="lines+markers", name=f"min_samples={int(min_samples)}",
                                 legendgroup=str(min_samples), showlegend=False, line=dict(color=color, width=2)), row=1, col=2)
    fig.update_xaxes(title_text="eps (skala FastDTW)")
    fig = style_figure(fig, None, 420, config)
    fig.update_annotations(font=dict(size=12, color=_cfg(config, "PLOT_INK")))
    return fig


def create_dbscan_grid_figure(dbscan_tuning: pd.DataFrame, best_params: dict | None = None, config: object | None = None) -> go.Figure:
    """Grid eps-kuantil × min_samples: Silhouette (kiri) dan persentase noise (kanan); ★ = terpilih."""
    frame = dbscan_tuning.copy()
    frame["Noise_%"] = 100 * (1 - frame["Coverage"])
    sil = frame.pivot_table(index="min_samples", columns="eps_quantile", values="Silhouette_FastDTW", aggfunc="first", dropna=False)
    noise = frame.pivot_table(index="min_samples", columns="eps_quantile", values="Noise_%", aggfunc="first", dropna=False)
    noise = noise.reindex(index=sil.index, columns=sil.columns)
    rows = [f"min_samples {int(v)}" for v in sil.index]
    cols = [f"q{v:.2f}" for v in sil.columns]
    star = np.full(sil.shape, "", dtype=object)
    if best_params:
        for i, ms in enumerate(sil.index):
            for j, q in enumerate(sil.columns):
                if int(ms) == int(best_params.get("min_samples", -1)) and np.isclose(q, best_params.get("eps_quantile", -1)):
                    star[i, j] = " ★"
    sil_text = np.where(sil.isna(), "–", sil.round(2).astype(str)) + star
    noise_text = np.where(noise.isna(), "–", noise.round(0).astype("Int64").astype(str) + "%") + star
    fig = make_subplots(rows=1, cols=2, subplot_titles=["Silhouette non-noise ↑ (– = hanya 1 cluster)", "Produk noise ↓"], horizontal_spacing=0.08)
    fig.add_trace(go.Heatmap(z=sil.to_numpy(), x=cols, y=rows, text=sil_text, texttemplate="%{text}", showscale=False, xgap=3, ygap=3,
                             colorscale=_scale(_cfg(config, "PLOT_SEQUENTIAL")), hoverongaps=False,
                             hovertemplate="%{y} · %{x}<br>Silhouette %{z:.3f}<extra></extra>"), row=1, col=1)
    fig.add_trace(go.Heatmap(z=noise.to_numpy(), x=cols, y=rows, text=noise_text, texttemplate="%{text}", showscale=False, xgap=3, ygap=3,
                             colorscale=_scale(_cfg(config, "PLOT_SEQUENTIAL_WARM")), zmin=0, zmax=100, hoverongaps=False,
                             hovertemplate="%{y} · %{x}<br>Noise %{z:.1f}%<extra></extra>"), row=1, col=2)
    fig.update_yaxes(autorange="reversed", showgrid=False)
    fig.update_yaxes(showticklabels=False, row=1, col=2)
    fig.update_xaxes(showgrid=False)
    fig = style_figure(fig, None, 300, config)
    fig.update_annotations(font=dict(size=12, color=_cfg(config, "PLOT_INK")))
    return fig


def create_silhouette_distribution_figure(detail: pd.DataFrame, config: object | None = None) -> go.Figure:
    """Histogram silhouette per produk, warna mengikuti cluster."""
    fig = go.Figure()
    for cluster_id in sorted(detail["Cluster"].unique()):
        fig.add_trace(go.Histogram(x=detail.loc[detail["Cluster"] == cluster_id, "Silhouette"], name=cluster_name(cluster_id),
                                   xbins=dict(start=-1, end=1, size=0.05), marker=dict(color=cluster_color(cluster_id, config)), opacity=0.7))
    fig.add_vline(x=0, line=dict(color=_cfg(config, "PLOT_MUTED"), width=1.2, dash="dot"))
    fig.update_layout(barmode="overlay")
    fig.update_xaxes(title_text="Silhouette individual · FastDTW", range=[-1, 1])
    fig.update_yaxes(title_text="Jumlah produk")
    return style_figure(fig, None, 340, config)


def create_silhouette_box_figure(detail: pd.DataFrame, config: object | None = None) -> go.Figure:
    """Box + titik silhouette individual menurut cluster; nilai negatif = lebih dekat ke cluster lain."""
    fig = go.Figure()
    has_code = "Product_Code" in detail.columns
    for cluster_id in sorted(detail["Cluster"].unique()):
        subset = detail.loc[detail["Cluster"] == cluster_id]
        color = cluster_color(cluster_id, config)
        fig.add_trace(go.Box(
            y=subset["Silhouette"], name=cluster_name(cluster_id), boxpoints="all", jitter=0.45, pointpos=0,
            marker=dict(color=_hex_to_rgba(color, 0.5), size=4), line=dict(color=_cfg(config, "PLOT_INK"), width=1.4),
            fillcolor="rgba(0,0,0,0)", customdata=subset["Product_Code"] if has_code else None,
            hovertemplate=("%{customdata}<br>" if has_code else "") + "Silhouette %{y:.3f}<extra></extra>",
        ))
    fig.add_hline(y=0, line=dict(color=_cfg(config, "PLOT_MUTED"), width=1.2, dash="dot"))
    fig.update_yaxes(title_text="Silhouette individual", range=[-1, 1])
    fig.update_layout(showlegend=False)
    return style_figure(fig, None, 340, config)


def create_pca_scatter_figure(pca_data: pd.DataFrame, config: object | None = None, product_codes: np.ndarray | None = None) -> go.Figure:
    """Scatter PCA dua dimensi (visual saja), warna mengikuti nomor cluster secara konsisten."""
    fig = go.Figure()
    for cluster_id in sorted(pca_data["Cluster"].unique()):
        mask = (pca_data["Cluster"] == cluster_id).to_numpy()
        subset = pca_data.loc[mask]
        codes = np.asarray(product_codes)[mask] if product_codes is not None else np.array([""] * mask.sum())
        fig.add_trace(go.Scattergl(
            x=subset["PC1"], y=subset["PC2"], mode="markers", name=cluster_name(cluster_id), customdata=codes,
            marker=dict(size=7, color=cluster_color(cluster_id, config), opacity=0.6, line=dict(color="#fff", width=0.6)),
            hovertemplate="%{customdata}<br>PC1 %{x:.2f} · PC2 %{y:.2f}<extra>" + cluster_name(cluster_id) + "</extra>",
        ))
    fig.update_xaxes(title_text="PC1 · visualisasi saja", showgrid=True, gridcolor=_cfg(config, "PLOT_GRID"))
    fig.update_yaxes(title_text="PC2 · visualisasi saja")
    return style_figure(fig, None, 400, config)


def create_weekly_diagnostics_figure(weekly_stats: pd.DataFrame, config: object | None = None) -> go.Figure:
    """Total penjualan mingguan dan persentase produk aktif pada panel terpisah (bukan sumbu ganda)."""
    palette = _cfg(config, "PLOT_PALETTE")
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                        subplot_titles=["Total penjualan mingguan", "Produk yang aktif (%)"])
    fig.add_trace(go.Scatter(x=weekly_stats["Minggu"], y=weekly_stats["Total_Penjualan"], mode="lines+markers", name="Total penjualan",
                             line=dict(color=palette[0], width=2.4), marker=dict(size=5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=weekly_stats["Minggu"], y=weekly_stats["Persen_Produk_Aktif"], mode="lines+markers", name="Produk aktif",
                             line=dict(color=palette[4], width=2.4), marker=dict(size=5)), row=2, col=1)
    fig.update_xaxes(title_text="Minggu", tickmode="array", tickvals=WEEK_TICKS, ticktext=[f"W{t}" for t in WEEK_TICKS], row=2, col=1)
    fig.update_yaxes(title_text="Volume", row=1, col=1)
    fig.update_yaxes(title_text="Produk aktif (%)", range=[0, 100], row=2, col=1)
    fig.add_vrect(x0=23.5, x1=26.5, fillcolor=_hex_to_rgba(palette[3], 0.08), line_width=0,
                  annotation_text="W24–W26", annotation_position="top left")
    fig = style_figure(fig, None, 560, config)
    fig.update_annotations(font=dict(size=12, color=_cfg(config, "PLOT_INK")))
    return fig


def build_dashboard_figures(
    W_original: np.ndarray,
    product_codes: np.ndarray,
    product_stats: pd.DataFrame,
    weekly_stats: pd.DataFrame,
    X_scaled: np.ndarray,
    labels: np.ndarray,
    profiles: pd.DataFrame,
    metrics: pd.DataFrame,
    ari: pd.DataFrame,
    nmi: pd.DataFrame,
    tuning_per_k: pd.DataFrame,
    tuning_dbscan: pd.DataFrame,
    silhouette_detail: pd.DataFrame,
    pca_data: pd.DataFrame,
    selected_product: str,
    config: object,
) -> dict[str, go.Figure]:
    """Kumpulan grafik lengkap untuk notebook/ekspor HTML (nama kunci lama dipertahankan)."""
    lookup = dict(zip(np.asarray(product_codes).astype(str), np.asarray(labels, dtype=int)))
    figures = {
        "01_weekly_sales": create_weekly_sales_figure(weekly_stats, config),
        "02_top_products": create_top_products_figure(product_stats, 10, config, lookup),
        "03_zero_weeks": create_zero_weeks_figure(product_stats, config),
    }
    figures.update({f"04_{name}": figure for name, figure in create_exploration_figures(
        product_stats, W_original, selected_product, config, labels=labels
    ).items()})
    figures["05_cluster_original"] = create_cluster_pattern_figure(W_original, labels, "Median volume · penjualan asli", config)
    figures["06_cluster_minmax"] = create_cluster_pattern_figure(X_scaled, labels, "Median nilai Min-Max · pola ternormalisasi", config)
    figures["07_cluster_contribution"] = create_cluster_contribution_figure(profiles, config)
    figures["08_metrics_comparison"] = create_metrics_comparison_figure(metrics, config)
    figures["09_coverage"] = create_coverage_figure(metrics, config)
    figures["10_ari_heatmap"] = create_similarity_heatmap(ari, "ARI", config)
    figures["11_nmi_heatmap"] = create_similarity_heatmap(nmi, "NMI", config)
    figures["12_tuning_k"] = create_tuning_k_figure(tuning_per_k, config)
    figures["13_dbscan_tuning"] = create_dbscan_tuning_figure(tuning_dbscan, config)
    figures["14_silhouette_distribution"] = create_silhouette_distribution_figure(silhouette_detail, config)
    figures["15_silhouette_by_cluster"] = create_silhouette_box_figure(silhouette_detail, config)
    figures["16_pca_scatter"] = create_pca_scatter_figure(pca_data, config, product_codes)
    figures["17_weekly_diagnostics"] = create_weekly_diagnostics_figure(weekly_stats, config)
    # Visual baru v2
    figures["18_pareto"] = create_pareto_figure(product_stats, config)
    figures["19_active_products"] = create_active_products_figure(weekly_stats, config)
    figures["20_sales_heatmap"] = create_sales_heatmap_figure(W_original, X_scaled, labels, product_codes, "minmax", config)
    figures["21_rank_heatmap"] = create_rank_heatmap_figure(metrics, config)
    figures["22_dbscan_grid"] = create_dbscan_grid_figure(tuning_dbscan, None, config)
    return figures


In [ ]:
%%writefile scripts/interactive_viz.py
"""Visual hasil klasterisasi interaktif bergaya R (factoextra / dbscan) dengan Plotly.

Padanan R:
* ``fviz_cluster``     → :func:`fviz_cluster_figure` (titik + convex hull/elips + centroid + medoid)
* ``fviz_silhouette``  → :func:`fviz_silhouette_figure`
* ``fviz_dend``        → :func:`fviz_dend_figure`
* ``fviz_nbclust``     → :func:`fviz_nbclust_figure`
* ``kNNdistplot``      → :func:`knn_distance_figure`
* ``dtwPlotTwoWay`` / ``dtwPlotDensity`` → :func:`dtw_alignment_figure`, :func:`dtw_cost_figure`
"""
from __future__ import annotations

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.cluster.hierarchy import dendrogram
from scipy.spatial import ConvexHull

from scripts.visualization import WEEK_TICKS, _cfg, _hex_to_rgba, cluster_color, cluster_name, style_figure

SYMBOLS = ["circle", "triangle-up", "square", "diamond", "cross", "x", "pentagon", "star-triangle-up"]


def _symbol(cluster_id: int) -> str:
    return "circle-open" if cluster_id == -1 else SYMBOLS[int(cluster_id) % len(SYMBOLS)]


def _ellipse(points: np.ndarray, level: float = 0.95, n: int = 80) -> np.ndarray | None:
    """Elips normal multivariat (seperti ``ellipse.type = "norm"`` di factoextra)."""
    if len(points) < 3:
        return None
    cov = np.cov(points.T)
    if not np.isfinite(cov).all() or np.linalg.det(cov) <= 1e-15:
        return None
    vals, vecs = np.linalg.eigh(cov)
    radius = np.sqrt(-2 * np.log(1 - level))  # chi-square 2 df
    t = np.linspace(0, 2 * np.pi, n)
    circle = np.c_[np.cos(t), np.sin(t)] * np.sqrt(np.maximum(vals, 0)) * radius
    return circle @ vecs.T + points.mean(axis=0)


def fviz_cluster_figure(
    coords: np.ndarray,
    explained: np.ndarray,
    labels: np.ndarray,
    codes: np.ndarray,
    center_xy: np.ndarray,
    cluster_ids: np.ndarray,
    medoids: np.ndarray,
    center_kind: str,
    shape: str = "hull",
    spider: bool = False,
    config: object | None = None,
    axis_name: str = "Dim",
) -> go.Figure:
    """Peta cluster 2D: titik per cluster (warna + simbol), hull/elips, centroid, dan medoid."""
    xy = np.asarray(coords)[:, :2]
    y = np.asarray(labels, dtype=int)
    codes = np.asarray(codes).astype(str)
    fig = go.Figure()
    for cluster_id in sorted(np.unique(y)):
        mask = y == cluster_id
        pts = xy[mask]
        color = cluster_color(cluster_id, config)
        if cluster_id != -1 and shape != "none" and len(pts) >= 3:
            outline = None
            if shape == "hull":
                try:
                    hull = ConvexHull(pts)
                    outline = pts[np.r_[hull.vertices, hull.vertices[0]]]
                except Exception:
                    outline = None
            else:
                outline = _ellipse(pts)
            if outline is not None:
                fig.add_trace(go.Scatter(x=outline[:, 0], y=outline[:, 1], mode="lines", fill="toself",
                                         fillcolor=_hex_to_rgba(color, 0.10), line=dict(color=color, width=1.2),
                                         hoverinfo="skip", showlegend=False, legendgroup=str(cluster_id)))
        if spider and cluster_id != -1:
            pos = int(np.flatnonzero(np.asarray(cluster_ids) == cluster_id)[0])
            cx, cy = center_xy[pos]
            sx = np.c_[pts[:, 0], np.full(len(pts), cx), np.full(len(pts), np.nan)].ravel()
            sy = np.c_[pts[:, 1], np.full(len(pts), cy), np.full(len(pts), np.nan)].ravel()
            fig.add_trace(go.Scattergl(x=sx, y=sy, mode="lines", line=dict(color=_hex_to_rgba(color, 0.18), width=0.7),
                                       hoverinfo="skip", showlegend=False, legendgroup=str(cluster_id)))
        fig.add_trace(go.Scattergl(
            x=pts[:, 0], y=pts[:, 1], mode="markers", name=f"{cluster_name(cluster_id)} · {mask.sum()}",
            legendgroup=str(cluster_id), customdata=codes[mask],
            marker=dict(size=7, color=color if cluster_id != -1 else _cfg(config, "PLOT_NOISE"), symbol=_symbol(cluster_id),
                        opacity=0.7, line=dict(color="#fff", width=0.5)),
            hovertemplate="<b>%{customdata}</b><br>" + cluster_name(cluster_id) + "<extra></extra>",
        ))
    if len(cluster_ids):
        kind_label = {"mean": "Centroid (rata-rata)", "dba": "Centroid DBA", "medoid": "Medoid"}[center_kind]
        fig.add_trace(go.Scatter(
            x=center_xy[:, 0], y=center_xy[:, 1], mode="markers+text", name=kind_label,
            text=[f"<b>{cluster_name(c)}</b>" for c in cluster_ids], textposition="top center",
            textfont=dict(size=13, color=_cfg(config, "PLOT_INK")),
            marker=dict(size=20, symbol="diamond" if center_kind != "medoid" else "star", color=[cluster_color(c, config) for c in cluster_ids],
                        line=dict(color=_cfg(config, "PLOT_INK"), width=2)),
            customdata=[codes[m] for m in medoids],
            hovertemplate="<b>%{text}</b><br>" + kind_label + "<br>Medoid: %{customdata}<extra></extra>",
        ))
        if center_kind != "medoid":
            fig.add_trace(go.Scatter(
                x=xy[medoids, 0], y=xy[medoids, 1], mode="markers", name="Medoid (produk representatif)",
                marker=dict(size=13, symbol="star", color="#fff", line=dict(color=_cfg(config, "PLOT_INK"), width=1.6)),
                customdata=codes[medoids], hovertemplate="Medoid %{customdata}<extra></extra>",
            ))
    ex = list(explained) + [0, 0]
    fig.update_xaxes(title_text=f"{axis_name}1 ({100 * ex[0]:.1f}%)", showgrid=True, gridcolor=_cfg(config, "PLOT_GRID"),
                     zeroline=True, zerolinecolor="#D7D2C4")
    fig.update_yaxes(title_text=f"{axis_name}2 ({100 * ex[1]:.1f}%)", zeroline=True, zerolinecolor="#D7D2C4")
    return style_figure(fig, None, 520, config)


def centroid_series_figure(
    X: np.ndarray,
    labels: np.ndarray,
    centers: np.ndarray,
    cluster_ids: np.ndarray,
    medoids: np.ndarray,
    codes: np.ndarray,
    center_kind: str,
    y_title: str,
    config: object | None = None,
    sample: int = 50,
    seed: int = 7,
) -> go.Figure:
    """Satu panel per cluster: anggota (garis tipis), pita IQR, centroid tebal, medoid putus-putus."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(labels, dtype=int)
    codes = np.asarray(codes).astype(str)
    ids = list(cluster_ids)
    cols = min(len(ids), 3) or 1
    rows = int(np.ceil(len(ids) / cols)) or 1
    fig = make_subplots(rows=rows, cols=cols, shared_yaxes=True, horizontal_spacing=0.04, vertical_spacing=0.16,
                        subplot_titles=[f"{cluster_name(c)} · {(y == c).sum()} produk" for c in ids])
    rng = np.random.default_rng(seed)
    weeks = np.arange(X.shape[1])
    center_name = {"mean": "Centroid (rata-rata)", "dba": "Centroid DBA", "medoid": "Medoid = centroid"}[center_kind]
    for index, c in enumerate(ids):
        r, col = index // cols + 1, index % cols + 1
        members = np.flatnonzero(y == c)
        color = cluster_color(c, config)
        pick = rng.choice(members, size=min(sample, len(members)), replace=False)
        xs = np.tile(np.r_[weeks, np.nan], len(pick))
        ys = np.c_[X[pick], np.full(len(pick), np.nan)].ravel()
        fig.add_trace(go.Scattergl(x=xs, y=ys, mode="lines", line=dict(color=_hex_to_rgba(color, 0.22), width=0.8),
                                   hoverinfo="skip", showlegend=False), row=r, col=col)
        q1, q3 = np.percentile(X[members], 25, axis=0), np.percentile(X[members], 75, axis=0)
        fig.add_trace(go.Scatter(x=weeks, y=q3, mode="lines", line=dict(width=0), hoverinfo="skip", showlegend=False), row=r, col=col)
        fig.add_trace(go.Scatter(x=weeks, y=q1, mode="lines", line=dict(width=0), fill="tonexty", fillcolor=_hex_to_rgba(color, 0.18),
                                 hoverinfo="skip", showlegend=index == 0, name="IQR anggota", legendgroup="iqr"), row=r, col=col)
        fig.add_trace(go.Scatter(x=weeks, y=centers[index], mode="lines", line=dict(color=_cfg(config, "PLOT_INK"), width=3),
                                 name=center_name, legendgroup="center", showlegend=index == 0,
                                 hovertemplate=f"{cluster_name(c)} W%{{x}}: %{{y:.3f}}<extra>{center_name}</extra>"), row=r, col=col)
        if center_kind != "medoid":
            fig.add_trace(go.Scatter(x=weeks, y=X[medoids[index]], mode="lines", line=dict(color=color, width=2, dash="dash"),
                                     name="Medoid", legendgroup="medoid", showlegend=index == 0,
                                     hovertemplate=f"Medoid {codes[medoids[index]]} W%{{x}}: %{{y:.3f}}<extra></extra>"), row=r, col=col)
        fig.update_xaxes(tickmode="array", tickvals=WEEK_TICKS, ticktext=[f"W{t}" for t in WEEK_TICKS], row=r, col=col)
    fig.update_yaxes(title_text=y_title, col=1)
    fig = style_figure(fig, None, 300 * rows + 40, config)
    fig.update_annotations(font=dict(size=12, color=_cfg(config, "PLOT_INK")))
    return fig


def centroid_overlay_figure(centers: np.ndarray, cluster_ids: np.ndarray, center_kind: str, y_title: str,
                            config: object | None = None) -> go.Figure:
    """Semua centroid dalam satu grafik untuk membandingkan bentuk pola antar-cluster."""
    fig = go.Figure()
    weeks = np.arange(np.asarray(centers).shape[1])
    for pos, c in enumerate(cluster_ids):
        fig.add_trace(go.Scatter(x=weeks, y=centers[pos], mode="lines+markers", name=cluster_name(c),
                                 line=dict(color=cluster_color(c, config), width=2.6), marker=dict(size=4),
                                 hovertemplate="W%{x}: %{y:.3f}<extra>" + cluster_name(c) + "</extra>"))
    fig.update_xaxes(title_text="Minggu", tickmode="array", tickvals=WEEK_TICKS, ticktext=[f"W{t}" for t in WEEK_TICKS])
    fig.update_yaxes(title_text=y_title)
    fig.update_layout(hovermode="x unified")
    return style_figure(fig, None, 340, config)


def fviz_silhouette_figure(silhouette: np.ndarray, labels: np.ndarray, codes: np.ndarray, config: object | None = None) -> go.Figure:
    """Plot silhouette gaya R: batang terurut per cluster dan garis rata-rata putus-putus."""
    s = np.asarray(silhouette, dtype=float)
    y = np.asarray(labels, dtype=int)
    codes = np.asarray(codes).astype(str)
    keep = np.isfinite(s) & (y != -1)
    frame = pd.DataFrame({"s": s[keep], "c": y[keep], "code": codes[keep]}).sort_values(["c", "s"], ascending=[True, False])
    frame["x"] = np.arange(len(frame))
    fig = go.Figure()
    for c, part in frame.groupby("c"):
        fig.add_trace(go.Bar(x=part["x"], y=part["s"], name=f"{cluster_name(c)} · rata² {part['s'].mean():.2f}",
                             marker=dict(color=cluster_color(c, config), line=dict(width=0)), customdata=part["code"],
                             hovertemplate="%{customdata}<br>Silhouette %{y:.3f}<extra></extra>"))
    if len(frame):
        fig.add_hline(y=frame["s"].mean(), line=dict(color="#C0392B", dash="dash", width=1.5),
                      annotation_text=f"rata-rata {frame['s'].mean():.3f}", annotation_position="top right")
    fig.update_layout(bargap=0)
    fig.update_xaxes(showticklabels=False, title_text="Produk (diurutkan per cluster)")
    fig.update_yaxes(title_text="Lebar silhouette Si", range=[min(-0.2, float(frame["s"].min()) - 0.05) if len(frame) else -1, 1])
    return style_figure(fig, None, 360, config)


def fviz_dend_figure(linkage_matrix: np.ndarray, labels: np.ndarray, k: int, config: object | None = None) -> go.Figure:
    """Dendrogram berwarna per cluster + garis potong (seperti ``fviz_dend(k = ...)``)."""
    Z = np.asarray(linkage_matrix)
    n = len(Z) + 1
    y = np.asarray(labels, dtype=int)
    leaves = dendrogram(Z, no_plot=True, count_sort=True)["leaves"]
    xpos = {leaf: 5.0 + 10.0 * i for i, leaf in enumerate(leaves)}
    height = {i: 0.0 for i in range(n)}
    node_cluster = {i: int(y[i]) for i in range(n)}
    segments: dict[int, tuple[list, list]] = {}
    for idx, (a, b, h, _) in enumerate(Z):
        a, b, node = int(a), int(b), n + idx
        xpos[node] = (xpos[a] + xpos[b]) / 2
        height[node] = float(h)
        ca, cb = node_cluster[a], node_cluster[b]
        node_cluster[node] = ca if ca == cb else -2
        bucket = segments.setdefault(node_cluster[node], ([], []))
        bucket[0].extend([xpos[a], xpos[a], xpos[b], xpos[b], None])
        bucket[1].extend([height[a], h, h, height[b], None])
    fig = go.Figure()
    for c, (xs, ds) in sorted(segments.items()):
        color = "#B9B3A4" if c == -2 else cluster_color(c, config)
        fig.add_trace(go.Scatter(x=xs, y=ds, mode="lines", line=dict(color=color, width=1.1), hoverinfo="skip",
                                 name="di atas potongan" if c == -2 else cluster_name(c), showlegend=c != -2))
    if k >= 2 and len(Z) >= k:
        cut = (Z[-k, 2] + Z[-k + 1, 2]) / 2 if k > 1 else Z[-1, 2]
        fig.add_hline(y=cut, line=dict(color="#C0392B", dash="dash", width=1.4), annotation_text=f"potong k = {k}",
                      annotation_position="top left")
    fig.update_xaxes(showticklabels=False, title_text=f"{n} produk (urutan daun dendrogram)", range=[0, 10 * n])
    fig.update_yaxes(title_text="Tinggi penggabungan (jarak)")
    return style_figure(fig, None, 420, config)


def fviz_nbclust_figure(rows: list[dict], chosen_k: int | None = None, config: object | None = None) -> go.Figure:
    """Metode siku (total jarak ke pusat) + rata-rata silhouette terhadap k."""
    frame = pd.DataFrame(rows)
    best_k = int(frame.loc[frame["Silhouette"].idxmax(), "k"]) if frame["Silhouette"].notna().any() else None
    fig = make_subplots(rows=1, cols=2, subplot_titles=["Metode siku · total jarak ke pusat ↓", "Rata-rata silhouette ↑"])
    primary = _cfg(config, "PLOT_PRIMARY")
    fig.add_trace(go.Scatter(x=frame["k"], y=frame["Within"], mode="lines+markers", line=dict(color=primary, width=2.4),
                             marker=dict(size=9), showlegend=False, hovertemplate="k=%{x}<br>%{y:,.1f}<extra></extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=frame["k"], y=frame["Silhouette"], mode="lines+markers", line=dict(color=primary, width=2.4),
                             marker=dict(size=9), showlegend=False, hovertemplate="k=%{x}<br>%{y:.3f}<extra></extra>"), row=1, col=2)
    for k, color, label in [(best_k, "#C0392B", "terbaik"), (chosen_k, _cfg(config, "PLOT_ACCENT"), "dipilih")]:
        if k is not None:
            for col in (1, 2):
                fig.add_vline(x=k, line=dict(color=color, dash="dash", width=1.3), row=1, col=col)
    fig.update_xaxes(title_text="Jumlah cluster k", dtick=1)
    fig = style_figure(fig, None, 340, config)
    fig.update_annotations(font=dict(size=12, color=_cfg(config, "PLOT_INK")))
    return fig


def knn_distance_figure(kdist: np.ndarray, eps: float | None, k: int, config: object | None = None) -> go.Figure:
    """Plot jarak tetangga ke-k terurut (``kNNdistplot``) dengan garis eps terpilih."""
    fig = go.Figure(go.Scatter(x=np.arange(len(kdist)), y=kdist, mode="lines", line=dict(color=_cfg(config, "PLOT_PRIMARY"), width=2.2),
                               hovertemplate="titik ke-%{x}<br>jarak %{y:.4g}<extra></extra>", name=f"jarak tetangga ke-{k}"))
    if eps is not None:
        fig.add_hline(y=eps, line=dict(color="#C0392B", dash="dash", width=1.5), annotation_text=f"eps = {eps:.4g}",
                      annotation_position="top left")
    fig.update_xaxes(title_text="Produk diurutkan menurut jarak")
    fig.update_yaxes(title_text=f"Jarak ke tetangga ke-{k}")
    return style_figure(fig, None, 320, config)


def distance_to_center_figure(dist_to_center: np.ndarray, labels: np.ndarray, codes: np.ndarray, config: object | None = None) -> go.Figure:
    """Sebaran jarak setiap produk ke pusat cluster-nya; ekor panjang = anggota pinggiran."""
    fig = go.Figure()
    y = np.asarray(labels, dtype=int)
    codes = np.asarray(codes).astype(str)
    for c in sorted(set(y.tolist()) - {-1}):
        mask = y == c
        color = cluster_color(c, config)
        fig.add_trace(go.Box(y=dist_to_center[mask], name=cluster_name(c), boxpoints="outliers", customdata=codes[mask],
                             marker=dict(color=color, size=4), line=dict(color=color), fillcolor=_hex_to_rgba(color, 0.15),
                             hovertemplate="%{customdata}<br>jarak %{y:.3f}<extra></extra>"))
    fig.update_yaxes(title_text="Jarak ke pusat cluster")
    fig.update_layout(showlegend=False)
    return style_figure(fig, None, 330, config)


def center_distance_figure(center_matrix: np.ndarray, cluster_ids: np.ndarray, config: object | None = None) -> go.Figure:
    """Jarak antar-pusat cluster (makin jauh = makin terpisah)."""
    names = [cluster_name(c) for c in cluster_ids]
    fig = go.Figure(go.Heatmap(z=center_matrix, x=names, y=names, text=np.round(center_matrix, 3), texttemplate="%{text}",
                               colorscale=[[0, "#F5F3EC"], [1, _cfg(config, "PLOT_PRIMARY")]], showscale=False, xgap=3, ygap=3,
                               hovertemplate="%{y} ↔ %{x}: %{z:.3f}<extra></extra>"))
    fig.update_yaxes(autorange="reversed", showgrid=False)
    return style_figure(fig, None, 260, config)


def cluster_heatmap_figure(X: np.ndarray, labels: np.ndarray, dist_to_center: np.ndarray, codes: np.ndarray,
                           config: object | None = None) -> go.Figure:
    """Peta panas anggota per cluster, diurutkan dari yang paling dekat ke pusat."""
    y = np.asarray(labels, dtype=int)
    d = np.where(np.isfinite(dist_to_center), dist_to_center, np.inf)
    order = np.lexsort((d, np.where(y == -1, 10 ** 6, y)))
    codes = np.asarray(codes).astype(str)
    fig = go.Figure(go.Heatmap(z=np.asarray(X)[order], x=[f"W{i}" for i in range(np.asarray(X).shape[1])], y=np.arange(len(order)),
                               text=np.repeat([[f"{codes[i]} · {cluster_name(y[i])}"] for i in order], np.asarray(X).shape[1], axis=1),
                               colorscale=[[i / 4, c] for i, c in enumerate(_cfg(config, "PLOT_SEQUENTIAL"))],
                               colorbar=dict(thickness=10, len=0.6), hovertemplate="%{text}<br>%{x}: %{z:.3f}<extra></extra>"))
    for b in np.flatnonzero(np.diff(y[order])) + 0.5:
        fig.add_hline(y=b, line=dict(color=_cfg(config, "PLOT_INK"), width=1.5))
    fig.update_yaxes(autorange="reversed", showticklabels=False, showgrid=False)
    fig.update_xaxes(tickmode="array", tickvals=[f"W{t}" for t in WEEK_TICKS], showgrid=False)
    return style_figure(fig, None, 480, config)


def dtw_alignment_figure(a: np.ndarray, b: np.ndarray, path: np.ndarray, name_a: str, name_b: str,
                         y_title: str, config: object | None = None) -> go.Figure:
    """Dua deret dengan garis penyejajaran DTW (``dtwPlotTwoWay``); deret B digeser agar terbaca."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    offset = (np.nanmax(a) - np.nanmin(b)) * 1.15 + 1e-9
    fig = go.Figure()
    xs, ys = [], []
    for i, j in path:
        xs += [i, j, None]
        ys += [a[i], b[j] - offset, None]
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="#B9B3A4", width=0.8), hoverinfo="skip", name="Pasangan minggu"))
    fig.add_trace(go.Scatter(x=np.arange(len(a)), y=a, mode="lines+markers", name=name_a, line=dict(color=cluster_color(0, config), width=2.2),
                             marker=dict(size=5), hovertemplate="W%{x}: %{y:.3f}<extra>" + name_a + "</extra>"))
    fig.add_trace(go.Scatter(x=np.arange(len(b)), y=b - offset, mode="lines+markers", name=f"{name_b} (digeser)",
                             line=dict(color=cluster_color(1, config), width=2.2), marker=dict(size=5), customdata=b,
                             hovertemplate="W%{x}: %{customdata:.3f}<extra>" + name_b + "</extra>"))
    fig.update_xaxes(title_text="Minggu", tickmode="array", tickvals=WEEK_TICKS, ticktext=[f"W{t}" for t in WEEK_TICKS])
    fig.update_yaxes(title_text=y_title, showticklabels=False)
    return style_figure(fig, None, 420, config)


def dtw_cost_figure(cost: np.ndarray, path: np.ndarray, window: int | None, name_a: str, name_b: str,
                    config: object | None = None) -> go.Figure:
    """Matriks biaya kumulatif DTW + jalur optimal + diagonal (Euclidean) dan pita Sakoe-Chiba."""
    C = np.where(np.isfinite(cost), cost, np.nan)
    n, m = C.shape
    fig = go.Figure(go.Heatmap(z=np.sqrt(C), colorscale=[[i / 4, c] for i, c in enumerate(_cfg(config, "PLOT_SEQUENTIAL"))],
                               colorbar=dict(title=dict(text="√biaya"), thickness=10, len=0.6),
                               hovertemplate=f"{name_a} W%{{y}} ↔ {name_b} W%{{x}}<br>√biaya kumulatif %{{z:.3f}}<extra></extra>"))
    fig.add_trace(go.Scatter(x=[0, m - 1], y=[0, n - 1], mode="lines", line=dict(color="#6B7770", dash="dot", width=1),
                             name="Diagonal (Euclidean)", hoverinfo="skip"))
    if window is not None:
        idx = np.arange(n)
        fig.add_trace(go.Scatter(x=np.r_[idx + window, (idx - window)[::-1]].clip(0, m - 1), y=np.r_[idx, idx[::-1]], mode="lines",
                                 line=dict(color=_cfg(config, "PLOT_ACCENT"), width=1, dash="dash"), name=f"Pita ±{window}", hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=path[:, 1], y=path[:, 0], mode="lines", line=dict(color="#C0392B", width=2.6), name="Jalur DTW optimal",
                             hovertemplate=f"{name_a} W%{{y}} ↔ {name_b} W%{{x}}<extra></extra>"))
    fig.update_xaxes(title_text=f"Minggu {name_b}", range=[-0.5, m - 0.5], showgrid=False)
    fig.update_yaxes(title_text=f"Minggu {name_a}", range=[-0.5, n - 0.5], showgrid=False, scaleanchor="x")
    return style_figure(fig, None, 460, config)


def membership_stack_figure(membership: np.ndarray, labels: np.ndarray, codes: np.ndarray, config: object | None = None) -> go.Figure:
    """Keanggotaan fuzzy semua produk sebagai batang bertumpuk (diurutkan per cluster & keyakinan)."""
    U = np.asarray(membership, dtype=float)
    y = np.asarray(labels, dtype=int)
    codes = np.asarray(codes).astype(str)
    order = np.lexsort((-U.max(axis=1), y))
    fig = go.Figure()
    for c in range(U.shape[1]):
        fig.add_trace(go.Bar(x=np.arange(len(order)), y=U[order, c], name=f"u {cluster_name(c)}", marker=dict(color=cluster_color(c, config), line=dict(width=0)),
                             customdata=codes[order], hovertemplate="%{customdata}<br>u = %{y:.3f}<extra>" + cluster_name(c) + "</extra>"))
    fig.update_layout(barmode="stack", bargap=0)
    fig.add_hline(y=0.5, line=dict(color=_cfg(config, "PLOT_INK"), dash="dot", width=1))
    fig.update_xaxes(showticklabels=False, title_text="Produk (urut cluster, lalu keyakinan)")
    fig.update_yaxes(title_text="Derajat keanggotaan", range=[0, 1])
    return style_figure(fig, None, 320, config)


In [ ]:
import importlib, numpy as np, plotly.io as pio
pio.renderers.default = "colab"
import config, scripts.interactive as ix, scripts.model_store as ms, scripts.visualization as V, scripts.interactive_viz as iv
for mod in (config, ix, ms, V, iv):
    importlib.reload(mod)
from scripts.data import build_product_statistics, build_weekly_statistics

W = sales[MINGGU].to_numpy(dtype=float)
CODES = sales["Product_Code"].astype(str).to_numpy()
STATS = build_product_statistics(sales, "Product_Code", MINGGU)
WEEKLY = build_weekly_statistics(sales, MINGGU)
try:
    import numba
    print("✅ numba", numba.__version__, "· thread:", getattr(getattr(numba, "config", None), "NUMBA_NUM_THREADS", "?"), "→ FastDTW asli dipakai")
except ImportError:
    print("⚠️ numba tidak ada → FastDTW diaproksimasi CDTW. Jalankan: !pip install numba")


## Sesi 3 — EDA singkat

In [ ]:
def show(fig):
    fig.update_layout(title=None); fig.show(config={"displaylogo": False})

z = 100 * (W == 0).mean()
print(f"Sel bernilai nol: {z:.1f}% · produk dengan ≥ 40 minggu nol: {int(((W == 0).sum(axis=1) >= 40).sum())}")
show(V.create_weekly_sales_figure(WEEKLY, config))
show(V.create_pareto_figure(STATS, config))
show(V.create_zero_weeks_figure(STATS, config))


**Tiga representasi.** *Data asli*: level & volume ikut menentukan jarak. *Min-Max per produk*: skala 0–1, hanya bentuk. *Scale (z-score per produk)*: rata-rata 0 dan simpangan baku 1 per produk, fokus pada naik-turun relatif.

In [ ]:
# Contoh satu produk pada ketiga representasi (inilah yang dibandingkan FastDTW)
contoh = "P1" if "P1" in set(CODES) else CODES[0]
i = list(CODES).index(contoh)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=[ix.REPRESENTATIONS[r] for r in config.REPRESENTATIONS])
for j, rep in enumerate(config.REPRESENTATIONS, start=1):
    fig.add_trace(go.Scatter(y=ix.represent(W, rep)[i], mode="lines", line=dict(color=config.PLOT_PALETTE[j - 1], width=2), name=ix.REPRESENTATIONS[rep]), row=1, col=j)
V.style_figure(fig, None, 300, config); show(fig)


## Sesi 4 — Membangun model
Setiap representasi: satu matriks FastDTW 811 × 811 (radius dari `config.py`), lalu 35 model: K-Means DBA, K-Medoids PAM, Fuzzy C-Means DBA, Fuzzy C-Medoids, dan Agglomerative (average & complete) untuk K = 2…6, serta DBSCAN dengan 5 nilai eps. Total 105 model.

In [ ]:
# ⏱️ Bagian terlama (±5–10 menit di Colab): 3 matriks FastDTW + 105 model
import time
t0 = time.time()
RESULTS = ms.build_model_results(W, CODES, config, cache_dir=f"{WORK}/cache", log=print)
print(f"\n✅ Selesai dalam {(time.time() - t0) / 60:.1f} menit · {len(RESULTS['runs'])} model dibangun")


## Sesi 5 — Evaluasi dan pemilihan model
**Aturan memilih** (dari `config.py`): minimal 2 cluster, Balance ≥ 0,5 (tidak ada cluster raksasa), noise ≤ 30%; lalu peringkat rata-rata Silhouette↑, DBI(MDS)↓, CHI(MDS)↑.

> Silhouette antar-representasi tidak sepenuhnya setara (ruang jarak berbeda). Pilih representasi sesuai tujuan: volume (Data asli) atau bentuk pola (Min-Max / Scale).

In [ ]:
from sklearn.metrics import adjusted_rand_score
table = ms.metrics_table(RESULTS)
best_rows = []
for rep, key in RESULTS["best"].items():
    if key:
        m = RESULTS["runs"][key]["metrics"]
        best_rows.append({"Representasi": ix.REPRESENTATIONS[rep], "Model terbaik": key, **{c: m[c] for c in ["Silhouette", "DBI_MDS", "CHI_MDS", "Balance", "Cluster", "Eta2_nol"]},
                          "Stabilitas": RESULTS["metadata"]["stability"].get(rep, {}).get("status", "–")})
print("MODEL TERBAIK PER REPRESENTASI (lolos guardrail, lalu peringkat Silhouette↑ · DBI↓ · CHI↑)")
display(pd.DataFrame(best_rows).round(3))
keys = {rep: k for rep, k in RESULTS["best"].items() if k}
ari = pd.DataFrame([[adjusted_rand_score(RESULTS["runs"][keys[a]]["labels"], RESULTS["runs"][keys[b]]["labels"]) for b in keys] for a in keys],
                   index=[ix.REPRESENTATIONS[r] for r in keys], columns=[ix.REPRESENTATIONS[r] for r in keys])
print("\nKESAMAAN PEMBAGIAN PRODUK ANTAR-REPRESENTASI (ARI; 1 = identik)")
display(ari.round(3))
for rep in config.REPRESENTATIONS:
    print(f"\n10 TERATAS · {ix.REPRESENTATIONS[rep]}")
    ranked = pd.DataFrame(RESULTS["_rankings"][rep])
    display(ranked.head(10).round(3) if "Skor_Ranking" in ranked else ranked.head(10))


## Sesi 6 — Visual model terbaik

In [ ]:
# Visual model terbaik tiap representasi (dashboard Streamlit menampilkan semuanya secara interaktif)
for rep, key in RESULTS["best"].items():
    if not key:
        print(f"{ix.REPRESENTATIONS[rep]}: tidak ada model yang lolos guardrail"); continue
    r = ms.run_to_result(RESULTS["runs"][key], RESULTS["spaces"][rep])
    sp = RESULTS["spaces"][rep]
    print(f"\n━━ {ix.REPRESENTATIONS[rep]} · {key} · Silhouette {r.metrics['Silhouette']:.3f}")
    show(iv.fviz_cluster_figure(np.array(sp["xy"]), np.array(sp["explained"]), r.labels, CODES, r.center_xy, r.cluster_ids, r.medoids, r.center_kind, "hull", False, config, "Dim"))
    X = ix.represent(W, rep)
    show(iv.centroid_series_figure(X, r.labels, r.centers, r.cluster_ids, r.medoids, CODES, r.center_kind, ix.REPRESENTATIONS[rep], config))
    show(iv.fviz_silhouette_figure(r.silhouette, r.labels, CODES, config))
    prof = pd.DataFrame(ix.cluster_profile_rows(r, W, CODES))
    display(prof[["Cluster", "Persona", "Tren", "Jumlah_Produk", "Porsi_Volume_%", "Produk_Representatif", "Rekomendasi"]].round(1))


## Sesi 7 — Ekspor untuk Streamlit

In [ ]:
# Simpan hasil untuk Streamlit: folder hasil_model + hasil_model.zip (di /content dan diunduh)
ZIP = "/content/hasil_model.zip"
ms.export_results(RESULTS, f"{WORK}/hasil_model", ZIP)
print("✅ Tersimpan:", ZIP, f"({os.path.getsize(ZIP) / 1e6:.1f} MB)")
print("   Isi:", ", ".join(sorted(os.listdir(f"{WORK}/hasil_model"))))
try:
    from google.colab import files
    files.download(ZIP)
except Exception:
    pass


## ➡️ Selanjutnya
Di Colab yang sama: **File → Upload notebook** → `02_jalankan_streamlit_colab.ipynb` → **Run all** → klik link dashboard.
`hasil_model.zip` yang terunduh juga bisa diunggah langsung di halaman awal dashboard, atau dimasukkan ke folder `hasil_model/` proyek bila di-deploy permanen.